# Grosmark & Buzsáki (2016) on CRCNS hc-11

Paper: Grosmark, A. D. and Buzsáki, G. (2016). Diversity in neural firing dynamics supports both rigid and learned hippocampal sequences. Science 351, 1440-1443, doi:10.1126/science.aad1935. The claims file (`claims/crcns-hc-11/10.1126-science.aad1935.yaml`) was written from the NIH author manuscript in PubMed Central (PMC4919122) and its supplement (Materials and Methods, Figures S1 to S22).

Dataset: CRCNS hc-11, doi:10.6080/K0862DC5, read from the public bucket at `https://crcnsarchive.s3-us-west-2.amazonaws.com/hc-11/live/`. Version: CRCNS data sets are not versioned, so the version is recorded as `unversioned`; the paper cites no version, and the `live/` folder is the only copy of the data files (the `old/` and `orig/` folders hold only earlier descriptions). The one file read is `data/NoveltySessInfoMatFiles.tar.gz`, MD5 `2666ee13c8c1e3609c08c6f4ba444d11` in the data set's `checksums.md5`, so that a later change can be detected. It is streamed and decompressed in memory; the eight `<session>_sessInfo.mat` files in it hold the spike times, cluster identities with putative cell types, position and the state epochs. For figs11 only, the `.clu` and `.fet` files of the eight `data/<session>.tar.gz` archives (0.8 to 2.3 GB each, 11.6 GB in all) are streamed in a background thread. The LFP (`*_eeg.tar.gz`, 5 to 10 GB each) and the waveforms (`*_spk.tar.gz`) are not read (see the deviations).

What differs from the paper's method throughout, as stated in `hc11.py` and the attempt record:

- Ripple events are the paper's population synchrony events without the condition that each holds the peak of an LFP-detected ripple: the LFP is only in tar.gz archives of 70 GB in all, which stream at about 23 MB/s (about 55 minutes), and the detector is not described.
- Offline states are the data set's own state epochs (reading S-a of the claims file): NREM, Drowsy (quiet waking) and Intermediate. On the maze, where the epochs mark only Wake, offline is time not running on the track (an assumption; 2D head speed under 5 cm/s is shown as a variant for Figure 1C).
- Readings the claims file lists are run where they apply: yoking Y-a, Y-b, Y-c; direction D-a, D-b; significance P-a, P-b. The claims that use the rigid and plastic groups use readings Y-a, D-a and P-a (an assumption).

| Claim | Tested here |
|---|---|
| place-cell-count | tested |
| fig1c-sequence-content | tested |
| figs5b-abs-r-above-null | tested |
| figs5e-duration-unchanged | tested |
| figs5f-more-participation-post | tested |
| figs9b-pcc-post-higher | tested |
| fig2b-strong-contributors | tested |
| fig2-pre-contributors-persist | tested |
| rigid-plastic-counts | tested |
| fig2d-high-dpcc-shuffle-abolishes-replay | tested (100 templates per step) |
| figs12-high-dpcc-exclusion | tested (200 null shuffles per step) |
| figs2-rigid-plastic-along-track | tested |
| fig3a-plastic-more-specific | tested |
| fig3b-plastic-specificity-increases | tested |
| figs13-outfield-rate-falls | tested |
| fig4a-rigid-fire-faster | tested |
| figs14-plastic-ripple-gain | tested |
| figs14-plastic-ripple-bursting | tested |
| figs16ab-plastic-synchrony-increase | tested |
| low-high-rate-split | tested |
| figs16c-low-rate-coding | tested |
| fig4c-coactivation-by-rate | tested |
| fig4d-slow-cells-replay | tested |
| fig4e-high-gain-cells-replay | tested |
| figs16de-more-ripples-post | ripple-event intervals tested; LFP-SWR intervals not |
| figs17-coactivity-preservation | tested |
| figs17d-rigid-plastic-coactivity | tested |
| figs18-temporal-bias | tested |
| figs11-cluster-quality-similar | blocked after verification round 2; tested in this revision as the analysis behind a proposed change |
| fig2e-plastic-removal | tested |
| figs19c-pcc-change-per-cell | tested |

The notebook took 38 to 76 minutes on the runners used, most of it the 1,000-shuffle replay and per-cell contribution analyses.


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")  # threads run the shuffles; one BLAS thread each
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource, warnings
import numpy as np
import pandas as pd
from scipy import stats
import hc11

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T_START = time.time()
SESSIONS, SESSINFO_MD5 = hc11.load_sessions()
print(f"streamed {len(SESSIONS)} sessions in {time.time() - T_START:.0f} s")
print(f"MD5 of the streamed file {SESSINFO_MD5}; checksums.md5 lists {hc11.SESSINFO_MD5}; match: {SESSINFO_MD5 == hc11.SESSINFO_MD5}")

# figs11: the .clu and .fet files of the eight <session>.tar.gz archives (11.6 GB) are streamed
# in one background thread while the analyses below run; the figs11 section collects them.
from concurrent.futures import ThreadPoolExecutor


def _cluster_quality_all():
    t0 = time.time()
    out = [hc11.cluster_quality(s) for s in SESSIONS]
    return out, time.time() - t0


CQ_FUTURE = ThreadPoolExecutor(1).submit(_cluster_quality_all)

In [ ]:
rows = []
for s in SESSIONS:
    pre, mz, post = s["pre"], s["mz"], s["post"]
    rows.append(dict(
        session=s["name"], animal=s["animal"], maze=s["maze"], pyr=len(s["pyr"]), int=len(s["intn"]),
        shanks=len(set(s["shank"].values())), spikes=s["n_spikes_total"],
        PRE_h=(pre[1] - pre[0]) / 3600, MAZE_min=(mz[1] - mz[0]) / 60, POST_h=(post[1] - post[0]) / 3600,
        offline_PRE_h=hc11.total(hc11.offline(s, "pre")) / 3600,
        offline_POST_h=hc11.total(hc11.offline(s, "post")) / 3600,
        last_spike_s=s["last_spike"], duration_s=s["duration"],
        pos_hz=1 / s["pos_dt"], x1d_max_m=np.nanmax(s["x1d"]), x1d_valid=np.mean(~np.isnan(s["x1d"])),
        run_min=s["running"].sum() * s["pos_dt"] / 60,
    ))
sanity = pd.DataFrame(rows)
print(sanity.round(3).to_string())
print("\nanimals:", sanity.groupby("animal").size().to_dict())
print("pyramidal cells:", sanity.pyr.sum(), " interneurons:", sanity["int"].sum())
print("mazes:", sanity.maze.value_counts().to_dict())
# checks of the description against the data
for s in SESSIONS:
    ids = set(s["spikes"])
    st = s["states"]
    over = s["last_spike"] - s["duration"]
    ep_ok = np.isclose(s["pre"][1], s["mz"][0]) and np.isclose(s["mz"][1], s["post"][0])
    print(f"{s['name']}: spike times sorted={s['sorted_times']}, epochs contiguous={ep_ok}, "
          f"spikes after sessDuration={over:.1f} s, every cluster is Pyr or Int={len(ids) == len(s['pyr']) + len(s['intn'])}, "
          f"maze states (s): " + ", ".join(f"{k} {hc11.total(hc11.clip_to(v, *s['mz'])):.0f}" for k, v in st.items()))
# position: linear mazes span the stem only (1.6 or 2 m); circular track spans the circumference minus the reward area
for s in SESSIONS:
    sp = s["speed2d"]
    print(f"{s['name']}: 1D range {np.nanmin(s['x1d']):.2f} to {np.nanmax(s['x1d']):.2f} m; "
          f"2D speed 5th/50th percentile {np.nanpercentile(sp, 5):.3f}/{np.nanpercentile(sp, 50):.3f} m/s; "
          f"MAZE offline (not running) {hc11.total(hc11.maze_immobile(s, 'notrun')) / 60:.1f} min, "
          f"(2D speed < 5 cm/s) {hc11.total(hc11.maze_immobile(s, '2d')) / 60:.1f} min")
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
# Clock check: spikes and position share the clock if place-field spikes are aligned to
# position. Decode position on the maze from 250 ms bins of running with each session's
# place maps (built on odd laps, tested on even samples), and repeat with the spike
# train shifted by 10 s; aligned data decode far better than shifted data.
def clock_check(s, shift):
    rng = np.random.default_rng(0)
    t, x, dt = s["pos_t"], s["x1d"], s["pos_dt"]
    m = s["running"]
    nb = int(np.ceil(np.nanmax(x) * 50))
    xb = np.clip(np.nan_to_num(x * 50).astype(int), 0, nb - 1)
    half = (np.floor((t - t[0]) / 30) % 2 == 0)
    occ = np.bincount(xb[m & half], minlength=nb) * dt
    cells = s["pyr"]
    maps, test_counts = [], []
    tt = t[m & ~half]
    for c in cells:
        sp = s["spikes"][c] + shift
        k = np.clip(np.searchsorted(t, sp), 0, len(t) - 1)
        ok = m[k] & half[k] & (np.abs(t[k] - sp) < dt)
        maps.append(hc11.smooth(np.bincount(xb[k[ok]], minlength=nb).astype(float)) / np.maximum(hc11.smooth(occ), 1e-9))
        test_counts.append(np.searchsorted(sp, tt + 0.125) - np.searchsorted(sp, tt - 0.125))
    F = np.array(maps) + 1e-3
    C = np.array(test_counts).T
    L = C @ np.log(F) - 0.25 * F.sum(0)
    dec = np.argmax(L, 1)
    return np.median(np.abs(dec - xb[m & ~half])) * 2  # cm

clock = pd.DataFrame([dict(session=s["name"], err_cm_aligned=clock_check(s, 0.0), err_cm_shift10s=clock_check(s, 10.0))
                      for s in SESSIONS])
print(clock.round(1).to_string())

In [ ]:
T0 = time.time()
RES = []
for i, s in enumerate(SESSIONS):
    t1 = time.time()
    RES.append(hc11.analyse_session(s, seed=100 + i))
    print(f"{s['name']}: {time.time() - t1:.0f} s, " + ", ".join(
        f"{d}: {len(v['events'])} events, {len(v['cells'])} place cells" for d, v in RES[-1]["rep"].items()),
        flush=True)
print(f"replay and PCC: {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
# Shared tables: replay events in the yoked windows, and per-unit PCC.
NAMES = [s["name"] for s in SESSIONS]
ANIMAL = {s["name"]: s["animal"] for s in SESSIONS}
YS, DS = ("a", "b", "c"), ("a", "b")


def replay_events(Y="a", D="a", mz="notrun", results=None):
    out = []
    for s, r in zip(SESSIONS, results or RES):
        pre, post = hc11.yoke(s, Y)
        for d, rep in r["rep"].items():
            ev = rep["events"].copy()
            ev["k"] = np.arange(len(ev))
            mid = (ev.start.values + ev.stop.values) / 2
            inmz = (ev.epoch == "MAZE") & (ev.offline if mz == "notrun" else ev.offline_2d)
            ev["win"] = np.where(hc11.in_intervals(mid, pre), "PRE",
                                 np.where(hc11.in_intervals(mid, post), "POST", np.where(inmz, "MAZE", "")))
            if D == "b":
                ev = ev[ev.assigned == d]
            ev["session"], ev["animal"], ev["dir"] = s["name"], s["animal"], d
            out.append(ev[ev.win != ""])
    return pd.concat(out, ignore_index=True)


def tukey_kramer_ranks(groups):
    """Kruskal-Wallis, then Tukey-Kramer comparisons of mean ranks (as MATLAB's multcompare)."""
    names = list(groups)
    H, p = stats.kruskal(*[groups[g] for g in names])
    allv = np.concatenate([groups[g] for g in names])
    rk = stats.rankdata(allv)
    N = len(allv)
    idx = np.cumsum([0] + [len(groups[g]) for g in names])
    mr = {g: rk[idx[i]:idx[i + 1]].mean() for i, g in enumerate(names)}
    pairs = {}
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            se = np.sqrt(N * (N + 1) / 12 * (1 / len(groups[a]) + 1 / len(groups[b])))
            q = abs(mr[a] - mr[b]) / se * np.sqrt(2)
            pairs[(a, b)] = (mr[a] - mr[b], float(stats.studentized_range.sf(q, len(names), 1e6)))
    return H, p, mr, pairs


def unit_table(Y="a", D="a", results=None):
    """One row per place unit (cell and running direction): PRE and POST PCC and tests."""
    rows = []
    for s, r in zip(SESSIONS, results or RES):
        pre, post = hc11.yoke(s, Y)
        for d, rep in r["rep"].items():
            ev = rep["events"]
            mid = (ev.start.values + ev.stop.values) / 2
            w = np.where(hc11.in_intervals(mid, pre), 1, np.where(hc11.in_intervals(mid, post), 2, 0))
            if D == "b":
                w = np.where(ev.assigned.values == d, w, 0)
            rz = ev.rz.values
            for c in rep["cells"]:
                evs, pc, msh = rep["pcc"][c]
                row = dict(session=s["name"], animal=s["animal"], cell=c, dir=d, shank=c // 100)
                for name, code in (("pre", 1), ("post", 2)):
                    m = w[evs] == code
                    row[f"n_{name}"] = int(m.sum())
                    row[f"pcc_{name}"] = float(np.mean(pc[m])) if m.any() else np.nan
                    row[f"pa_{name}"] = hc11.wilcoxon_1s(pc[m]) if m.sum() > 1 else np.nan
                    row[f"pb_{name}"] = (float(stats.ranksums(rz[evs[m]], msh[m], alternative="greater").pvalue)
                                         if m.sum() > 1 else np.nan)
                    row[f"_pcc_{name}"] = pc[m]
                rows.append(row)
    u = pd.DataFrame(rows)
    u["dpcc"] = u.pcc_post - u.pcc_pre
    return u


def classify(u, P="a"):
    """Strongly contributing units, and rigid and plastic within each session and direction [M]."""
    u = u.copy()
    u["sig_pre"] = u[f"p{P}_pre"] < 0.05
    u["sig_post"] = u[f"p{P}_post"] < 0.05
    u["strong"] = u.sig_pre | u.sig_post
    u["group"] = ""
    for (_, _), g in u[u.strong].groupby(["session", "dir"]):
        g = g.sort_values("dpcc")
        n = len(g)
        idx = list(g.index)
        if n % 2 == 1:
            idx.pop(n // 2)
        h = len(idx) // 2
        u.loc[idx[:h], "group"] = "rigid"
        u.loc[idx[h:], "group"] = "plastic"
    return u


EV = {(Y, D): replay_events(Y, D) for Y in YS for D in DS}
UNITS = {(Y, D): unit_table(Y, D) for Y in YS for D in DS}
print({k: len(v) for k, v in EV.items()})
print(EV[("a", "a")].groupby(["session", "dir", "win"]).size().unstack().to_string())

In [ ]:
# Ripple-event selection step by step, and events per hour of NREM in PRE and POST. The
# paper's events also hold an LFP-detected ripple peak, which is not applied here.
rows, ALL_GAPS = [], []
for s, r in zip(SESSIONS, RES):
    t = r["all_events"]
    nrem = np.vstack([hc11.clip_to(s["states"]["NREM"], *s["pre"]), hc11.clip_to(s["states"]["NREM"], *s["post"])])
    dur_ok = (t.dur >= 0.05) & (t.dur <= 0.5)
    off_ok = dur_ok & (t.offline | t.offline_2d)
    five = off_ok & (t.npyr >= 5)
    in_nrem = hc11.in_intervals(((t.start + t.stop) / 2).values, nrem)
    nrem_h = hc11.total(nrem) / 3600
    per_h = (five & in_nrem).sum() / nrem_h
    mids = np.sort(((t.start + t.stop) / 2).values[five & in_nrem])
    k = np.searchsorted(nrem[np.argsort(nrem[:, 0]), 0], mids, side="right")
    gaps = np.diff(mids)[(np.diff(k) == 0) & (np.diff(mids) < 60)]  # within one NREM interval, under 60 s
    ALL_GAPS.append(gaps)
    rows.append(dict(session=s["name"], candidates=len(t), dur_50_500ms=int(dur_ok.sum()), offline=int(off_ok.sum()),
                     five_pyr=int(five.sum()), **{f"replay_{d}": len(v["events"]) for d, v in r["rep"].items()},
                     NREM_h=nrem_h, in_NREM=int((five & in_nrem).sum()), per_NREM_h=per_h, mean_interval_NREM_s=3600 / per_h,
                     median_interval_NREM_s=np.median(gaps)))
sel = pd.DataFrame(rows)
print(sel.to_string(float_format=lambda v: f"{v:.2f}"))
print("replay_<dir>: events kept for each template (at least 100 ms and 10% or 5 place cells firing)")
print(f"all sessions: {sel.in_NREM.sum()} events in {sel.NREM_h.sum():.1f} h of NREM, {sel.in_NREM.sum() / sel.NREM_h.sum():.0f} "
      f"per hour, mean interval {3600 * sel.NREM_h.sum() / sel.in_NREM.sum():.2f} s; median interval between consecutive events "
      f"within one NREM interval (under 60 s) {np.median(np.concatenate(ALL_GAPS)):.2f} s")


Ripple-event selection: the candidates fall from 7,823 to 19,662 per session to 5,771 to 12,511 after the duration, offline and five-cell conditions. In NREM there are 1,855 events per hour over all sessions (1,771 to 1,928 per session), a mean interval of 1.94 s; the median interval between consecutive events within one NREM interval (under 60 s apart) is 1.19 s, against the paper's median ripple-event intervals of 3 and 2.47 s. The LFP condition, not applied here, would remove events; how many cannot be said without the LFP.

## place-cell-count

> Of the 562 well isolated CA1 pyramidal cells recorded in the eight sessions, 491 had place fields on the novel maze, counted separately for each running direction on the linear mazes, which is 54% of pyramidal cells by running direction.

Source: Results, paragraph 2; Materials and Methods, Animals, Surgery and Data Collection; Figure S2 legend; Figure S19 legend.

Criterion (claims file): Pyramidal cells as the data set's putative cell types label them. Place fields by the place-field method of the opening comment, per running direction on the linear mazes and once on the circular maze. The denominator of 54% is open; readings (a) pyramidal cells times the number of running directions of their session (2 on linear mazes, 1 on the circular maze); (b) the 562 pyramidal cells. Reproduced when the counts are within tolerance under one reading.

In [ ]:
pc_rows = []
for s, r in zip(SESSIONS, RES):
    for d, pm in r["pm"].items():
        pc_rows.append(dict(session=s["name"], dir=d, pyr=len(s["pyr"]),
                            place=sum(1 for c in s["pyr"] if pm["fields"][c])))
pct = pd.DataFrame(pc_rows)
print(pct.to_string())
n_place = pct.place.sum()
n_pyr = sum(len(s["pyr"]) for s in SESSIONS)
den_a = pct.pyr.sum()
print(f"place units (cell x direction): {n_place}  [expected 491 +- 49.1]")
print(f"pyramidal cells: {n_pyr}  [expected 562 +- 56.2]")
print(f"reading (a), place units / (pyramidal cells x directions) = {n_place}/{den_a} = {100 * n_place / den_a:.1f}%")
print(f"reading (b), place units / pyramidal cells = {n_place}/{n_pyr} = {100 * n_place / n_pyr:.1f}%  [expected 54 +- 5.4]")
print("distinct cells with a field in either direction:",
      sum(len({c for d, pm in r['pm'].items() for c in s['pyr'] if pm['fields'][c]}) for s, r in zip(SESSIONS, RES)))
print("per animal:", pct.assign(animal=pct.session.str.split('_').str[0]).groupby('animal')[['place', 'pyr']].sum().to_dict('index'))

**Result: partially reproduced, resting on an assumption.** The data set holds 562 pyramidal cells (562 expected). With the 5 cm kernel read as an SD, 565 place units (cell x direction) have a field, 62.1% of the 910 pyramidal cells x directions, above 491 +- 49.1 and 54 +- 5.4%; the paper's 491 is 54% of 910, so reading (a) is the paper's denominator, and reading (b) gives 100.5%. With the kernel read as a full width at half maximum, 504 units and 55.4% are within tolerance. The kernel width is open; the SD reading is adopted.

In [ ]:
# Variant: the 5 cm kernel read as a full width at half maximum (SD 2.12 cm) in place of an SD.
_saved = hc11.SMOOTH_CM
hc11.SMOOTH_CM = 5.0 / 2.3548
n_fwhm = 0
for i, s in enumerate(SESSIONS):
    pm = hc11.place_maps(s, np.random.default_rng(500 + i))
    n_fwhm += sum(1 for d in pm for c in s["pyr"] if pm[d]["fields"][c])
hc11.SMOOTH_CM = _saved
print(f"variant, kernel as FWHM: {n_fwhm} place units, {100 * n_fwhm / den_a:.1f}% of pyramidal cells x directions, "
      f"{100 * n_fwhm / n_pyr:.1f}% of pyramidal cells")

## fig1c-sequence-content

> Ripple-event sequence scores (rZ) are above chance in PRE, MAZE and POST, highest on the MAZE, and higher in POST than in PRE sleep (replay).

Source: Figure 1C; Figure S5 legend; Figure S6 (per animal); Results, paragraph 2.

Criterion (claims file): Events, decoding and rZ by the ripple-event and Bayesian replay methods of the opening comment. Tests from the Figure 1C legend, a sign-rank test of rZ against zero within each epoch, and a Kruskal-Wallis test across PRE, MAZE and POST followed by Tukey-Kramer tests. Reproduced when rZ is above zero in all three epochs (p < 0.05), POST rZ exceeds PRE rZ and MAZE exceeds both (Tukey-Kramer p < 0.05), under all readings of Y, L, S and D; partially reproduced when POST exceeds PRE but another part fails. The Figure 1C panel values are not available here, so no level is fixed; Figure S6 shows MAZE highest in each of the four animals and POST above PRE in three of them (not in animal B), which the per-animal results are compared with. The time-bin-shuffle rZ of Figure S22B is shown as a variant.

In [ ]:
fig1c = []
for Y in YS:
    for D in DS:
        for mz in ("notrun", "2d"):
            ev = EV[(Y, D)] if mz == "notrun" else replay_events(Y, D, mz)
            g = {w: ev.rz[ev.win == w].dropna().values for w in ("PRE", "MAZE", "POST")}
            H, pkw, mr, pairs = tukey_kramer_ranks(g)
            row = dict(Y=Y, D=D, maze_offline=mz)
            for w in g:
                row[f"n_{w}"] = len(g[w])
                row[f"mean_{w}"] = g[w].mean()
                row[f"p0_{w}"] = hc11.wilcoxon_1s(g[w])
            row["p_KW"] = pkw
            row["TK_POST>PRE"] = pairs[("PRE", "POST")][1] if pairs[("PRE", "POST")][0] < 0 else 1.0
            row["TK_MAZE>PRE"] = pairs[("PRE", "MAZE")][1] if pairs[("PRE", "MAZE")][0] < 0 else 1.0
            row["TK_MAZE>POST"] = pairs[("MAZE", "POST")][1] if pairs[("MAZE", "POST")][0] > 0 else 1.0
            fig1c.append(row)
fig1c = pd.DataFrame(fig1c)
print(fig1c.to_string(float_format=lambda v: f"{v:.3g}"))
ok = ((fig1c[["p0_PRE", "p0_MAZE", "p0_POST"]] < 0.05).all(axis=1) & (fig1c[["TK_POST>PRE", "TK_MAZE>PRE", "TK_MAZE>POST"]] < 0.05).all(axis=1))
print(f"\nreadings where every condition holds: {ok.sum()} of {len(ok)}; POST > PRE (Tukey-Kramer p < 0.05) in "
      f"{(fig1c['TK_POST>PRE'] < 0.05).sum()} of {len(ok)}")

In [ ]:
# Per animal (Figure S6), readings Y-a, D-a.
ev = EV[("a", "a")]
pa = []
for a, g in ev.groupby("animal"):
    gg = {w: g.rz[g.win == w].dropna().values for w in ("PRE", "MAZE", "POST")}
    pa.append(dict(animal=a, **{f"mean_{w}": v.mean() for w, v in gg.items()}, **{f"n_{w}": len(v) for w, v in gg.items()},
                   p_POST_gt_PRE=float(stats.ranksums(gg["POST"], gg["PRE"], alternative="greater").pvalue)))
pa = pd.DataFrame(pa)
pa["MAZE_highest"] = (pa.mean_MAZE > pa.mean_POST) & (pa.mean_MAZE > pa.mean_PRE)
pa["POST_above_PRE"] = pa.mean_POST > pa.mean_PRE
print(pa.to_string(float_format=lambda v: f"{v:.3g}"))
print("per session (Y-a, D-a):")
print(ev.groupby(["session", "win"]).rz.mean().unstack().round(3).to_string())
print("rZ range:", np.nanmin(ev.rz).round(2), "to", np.nanmax(ev.rz).round(2))

def within_session(ev, col):
    zs = []
    for (sn, d), g in ev.groupby(["session", "dir"]):
        a, b = g[col][g.win == "PRE"].dropna(), g[col][g.win == "POST"].dropna()
        st_ = stats.ranksums(b, a)
        zs.append(dict(session=sn, dir=d, n_PRE=len(a), n_POST=len(b), mean_PRE=a.mean(), mean_POST=b.mean(),
                       z=st_.statistic, p=st_.pvalue))
    zs = pd.DataFrame(zs)
    print(zs.to_string(float_format=lambda v: f"{v:.3g}"))
    Z = zs.z.sum() / np.sqrt(len(zs))
    print(f"stratified POST > PRE (Stouffer, equal weights): z = {Z:.2f}, one-sided p = {stats.norm.sf(Z):.3g}; "
          f"templates with POST mean above PRE: {(zs.mean_POST > zs.mean_PRE).sum()} of {len(zs)}")
# Within session (Y-a, D-a): POST against PRE rZ per template, and a stratified test.
within_session(ev, "rz")


**Result: reproduced, resting on an assumption.** Under all 12 combinations of the Y, D and MAZE-offline readings, mean rZ is above zero in PRE (0.173 to 0.208), MAZE (0.506 to 0.561) and POST (0.283 to 0.304), all sign-rank p < 1e-30, and the Kruskal-Wallis test with Tukey-Kramer comparisons gives MAZE > POST > PRE (largest p 3.2e-10, Y-c, D-b, 2D). By animal POST exceeds PRE in all four (rank-sum p 1.6e-19 to 0.020), and MAZE is highest in three; in Buddy POST (0.504) exceeds MAZE (0.484). The paper's Figure S6 has MAZE highest in all four and POST above PRE in three. Within session (Y-a, D-a), POST mean rZ exceeds PRE in 13 of 13 templates; the stratified test gives z = 10.3, p = 2.5e-25. The L readings were not run (the LFP was not read), so the criterion's readings of L are not covered.

## figs5b-abs-r-above-null

> Absolute weighted correlations of ripple events exceed those of the shuffled null in PRE, MAZE and POST, and the POST minus PRE increase is larger in the observed than in the null data.

Source: Figure S5B.

Criterion (claims file): Events and shuffles as in fig1c-sequence-content. Per the Figure S5B legend, observed against null |r| within each epoch by rank-sum test, and the POST - PRE change of mean |r| tested by bootstrap. Reproduced when observed |r| exceeds null |r| in all three epochs at p < 0.05 and the observed POST - PRE change exceeds the null change (bootstrap 95% interval of the difference above zero), under the readings of fig1c-sequence-content.

In [ ]:
rng5b = np.random.default_rng(51)
f5b = []
for Y in YS:
    for D in DS:
        ev = EV[(Y, D)]
        row = dict(Y=Y, D=D)
        for w in ("PRE", "MAZE", "POST"):
            g = ev[ev.win == w]
            row[f"obs_{w}"] = np.abs(g.r).mean()
            row[f"null_{w}"] = g.null_mu.mean()
            row[f"p_{w}"] = (float(stats.ranksums(np.abs(g.r), g.null_one, alternative="greater").pvalue)
                             if "null_one" in g else np.nan)
            row[f"p_{w}_vs_null_mean"] = float(stats.ranksums(np.abs(g.r), g.null_mu, alternative="greater").pvalue)
        a = ev[ev.win == "PRE"]
        b = ev[ev.win == "POST"]
        ao, an = np.abs(a.r).values, a.null_mu.values
        bo, bn = np.abs(b.r).values, b.null_mu.values
        boot = []
        for _ in range(2000):
            i = rng5b.integers(0, len(a), len(a))
            j = rng5b.integers(0, len(b), len(b))
            boot.append((bo[j].mean() - ao[i].mean()) - (bn[j].mean() - an[i].mean()))
        row["dObs"] = bo.mean() - ao.mean()
        row["dNull"] = bn.mean() - an.mean()
        row["ci_lo"], row["ci_hi"] = np.percentile(boot, [2.5, 97.5])
        f5b.append(row)
f5b = pd.DataFrame(f5b)
print(f5b.to_string(float_format=lambda v: f"{v:.4g}"))

**Result: reproduced, resting on an assumption.** Compared with one null |r| per event (from the first random template), observed |r| exceeds the null in PRE, MAZE and POST under every reading (largest p 4.4e-31), and the POST - PRE change of mean |r| exceeds the null's (difference observed minus null, bootstrap 95% interval 0.0076 to 0.0113 under Y-a, D-a, above zero under every reading). Compared with each event's mean null |r|, the PRE comparison fails (p = 1 under every reading), as means have less spread than single values; that comparison is shown, not adopted.

## figs5e-duration-unchanged

> Ripple-event duration did not differ between PRE and POST (median 183 ms against 186 ms).

Source: Figure S5E legend.

Criterion (claims file): A claim of absence, judged by the paper's rank-sum test of PRE against POST event durations at p < 0.05. Which events are plotted is open; readings (a) all ripple events of the opening comment; (b) the events used for Bayesian replay (at least 100 ms, enough place cells). Reproduced when the test is not significant and both medians are within tolerance under one reading; the size of the difference is reported either way.

In [ ]:
f5e = []
for Y in YS:
    rows = []
    for s, r in zip(SESSIONS, RES):
        pre, post = hc11.yoke(s, Y)
        rp = r["ripples"]
        mid = (rp.start.values + rp.stop.values) / 2
        rows.append(pd.DataFrame(dict(dur=rp.dur.values * 1000,
                                      win=np.where(hc11.in_intervals(mid, pre), "PRE",
                                                   np.where(hc11.in_intervals(mid, post), "POST", "")),
                                      start=rp.start.values, session=s["name"])))
    allr = pd.concat(rows)
    ev = EV[(Y, "a")].drop_duplicates(["session", "start"])
    for reading, tab in (("a all ripple events", allr), ("b replay events", ev.assign(dur=ev.dur * 1000))):
        a, b = tab.dur[tab.win == "PRE"], tab.dur[tab.win == "POST"]
        f5e.append(dict(Y=Y, reading=reading, n_PRE=len(a), n_POST=len(b), median_PRE=a.median(),
                        median_POST=b.median(), p_ranksum=float(stats.ranksums(a, b).pvalue)))
f5e = pd.DataFrame(f5e)
print(f5e.to_string(float_format=lambda v: f"{v:.4g}"))
print("expected medians 183 +- 18.3 (PRE) and 186 +- 18.6 ms (POST)")

**Result: reproduced.** Under reading (b), the replay events, median durations are 173 ms in PRE and 173 ms in POST (within 183 +- 18.3 and 186 +- 18.6) and the rank-sum test finds no difference (p = 0.72, 0.91 and 0.48 under Y-a, Y-b, Y-c). Under reading (a), all ripple events, the medians are 144 and 143 ms and the test is significant (p = 0.0004), with a 1 ms difference.

## figs5f-more-participation-post

> A greater percentage of place cells participated (fired at least one spike) in POST than in PRE ripple events.

Source: Figure S5F legend.

Criterion (claims file): Per event, the percentage of the session's place cells (per running direction template under reading D-a) firing at least one spike, on the events of fig1c-sequence-content. The Figure S5 legend names no test for panel F; the rank-sum test it names for panel E is used. Reproduced when POST exceeds PRE at p < 0.05.

In [ ]:
f5f = []
for Y in YS:
    ev = EV[(Y, "a")]
    a, b = 100 * ev.frac_part[ev.win == "PRE"], 100 * ev.frac_part[ev.win == "POST"]
    f5f.append(dict(Y=Y, n_PRE=len(a), n_POST=len(b), mean_PRE=a.mean(), mean_POST=b.mean(),
                    median_PRE=a.median(), median_POST=b.median(),
                    p_POST_gt_PRE=float(stats.ranksums(b, a, alternative="greater").pvalue)))
print(pd.DataFrame(f5f).to_string(float_format=lambda v: f"{v:.4g}"))
ev = EV[("a", "a")]
print(ev.groupby(["session", "win"]).frac_part.mean().unstack()[["PRE", "POST"]].mul(100).round(1).to_string())
# Within session (Y-a, D-a): POST against PRE participation per template, and a stratified test.
within_session(EV[("a", "a")], "frac_part")


**Result: reproduced.** A greater percentage of place cells fires in POST than in PRE events (mean 32.99 against 31.36%, rank-sum p = 3.1e-88 under Y-a; p < 1e-86 under Y-b and Y-c). Six of eight sessions go the same way; Buddy_06272013 and Cicero_09012014 do not. Within session and template, POST exceeds PRE in 9 of 13 templates; the stratified test gives z = 8.2, p = 1.0e-16, so pooling sessions with unequal event counts inflates the pooled p but does not create the effect.

## figs9b-pcc-post-higher

> Place cells contribute more to POST than to PRE sequence content (median PCC -0.011 in PRE, 0.183 in POST), and the median change dPCC is 0.159.

Source: Figure S9B (values read from the panel titles).

Criterion (claims file): PCC by the method of the opening comment, for all place cells. Panel titles print "p ~ 0", recorded as 0. The tolerance on the PRE median is not 10% of a value near zero; it is about 10% of the gap between the PRE and POST medians (0.194). Reproduced when POST PCC exceeds PRE PCC (rank-sum p < 0.05), dPCC exceeds zero (signed-rank p < 0.05), and the three medians are within tolerance, under readings Y and D; partially reproduced when the tests hold but a median falls outside its tolerance.

In [ ]:
f9 = []
for Y in YS:
    for D in DS:
        u = UNITS[(Y, D)]
        v = u.dropna(subset=["pcc_pre", "pcc_post"])
        f9.append(dict(Y=Y, D=D, units=len(u), units_with_both=len(v), median_pre=v.pcc_pre.median(),
                       median_post=v.pcc_post.median(), median_dpcc=v.dpcc.median(),
                       p_ranksum_post_gt_pre=float(stats.ranksums(v.pcc_post, v.pcc_pre, alternative="greater").pvalue),
                       p_signrank_dpcc_gt0=hc11.wilcoxon_1s(v.dpcc)))
f9 = pd.DataFrame(f9)
print(f9.to_string(float_format=lambda v: f"{v:.4g}"))
print("expected medians: PRE -0.011 +- 0.02, POST 0.183 +- 0.0183, dPCC 0.159 +- 0.0159")
u = UNITS[("a", "a")]
print("per session (Y-a, D-a), median PCC:")
print(u.groupby("session")[["pcc_pre", "pcc_post", "dpcc"]].median().round(3).to_string())
print("PCC range (Y-a, D-a):", np.nanmin(u[["pcc_pre", "pcc_post"]].values).round(2), "to",
      np.nanmax(u[["pcc_pre", "pcc_post"]].values).round(2), "; units without PRE or POST events:",
      int(u[["pcc_pre", "pcc_post"]].isna().any(axis=1).sum()))

**Result: partially reproduced, resting on an assumption.** POST PCC exceeds PRE PCC (rank-sum p = 2.5e-06) and dPCC exceeds zero (signed-rank p = 1.9e-14) under Y-a, D-a, and both tests hold under all six readings (largest p 1.3e-04). The medians are smaller than the paper's: PRE 0.012 (expected -0.011 +- 0.02; 0.0035 to 0.0202 over readings, inside under Y-b, D-b and Y-c, D-b), POST 0.102 (0.183 +- 0.0183) and dPCC 0.114 (0.159 +- 0.0159).

## fig2b-strong-contributors

> 216 place cells contributed significantly to PRE or POST sequence content; 97 in PRE, 190 in POST, 71 in both, 26 in PRE only and 119 in POST only.

Source: Results, paragraph 2; Figure 2B; Figure S19A (counts read from the panel).

Criterion (claims file): Counts of place cells (per running direction) with per-event PCC significant at p < 0.05 one-tailed, under readings P-a and P-b of the opening comment. Reproduced when all four counts are within tolerance under one reading; partially reproduced when the total is within tolerance but a subset count is not.

In [ ]:
f2b = []
for Y in YS:
    for D in DS:
        for P in ("a", "b"):
            c = classify(UNITS[(Y, D)], P)
            f2b.append(dict(Y=Y, D=D, P=P, strong=int(c.strong.sum()), sig_pre=int(c.sig_pre.sum()),
                            sig_post=int(c.sig_post.sum()), both=int((c.sig_pre & c.sig_post).sum()),
                            pre_only=int((c.sig_pre & ~c.sig_post).sum()), post_only=int((~c.sig_pre & c.sig_post).sum()),
                            pct_pre_also_post=100 * (c.sig_pre & c.sig_post).sum() / max(c.sig_pre.sum(), 1),
                            rigid=int((c.group == "rigid").sum()), plastic=int((c.group == "plastic").sum())))
f2b = pd.DataFrame(f2b)
exp = dict(strong=(216, 21.6), sig_pre=(97, 9.7), sig_post=(190, 19), both=(71, 7.1))
f2b["all_four_in_tol"] = np.all([np.abs(f2b[k] - v) <= t for k, (v, t) in exp.items()], axis=0)
print(f2b.drop(columns=["pct_pre_also_post", "rigid", "plastic"]).to_string())
print("expected: strong 216 +- 21.6, PRE 97 +- 9.7, POST 190 +- 19, both 71 +- 7.1")

**Result: partially reproduced.** Under P-a with D-b the total is within tolerance and the PRE count is not: Y-a gives 222 strongly contributing units (216 +- 21.6), 109 in PRE (97 +- 9.7), 184 in POST (190 +- 19) and 71 in both (71 +- 7.1); Y-b and Y-c are alike (PRE 119 and 109). Under P-a with D-a the counts are larger (247, 134, 212, 99), and under P-b far smaller (22 to 39 in all).

## fig2-pre-contributors-persist

> The majority (73%) of neurons strongly contributing to PRE sequences also contributed to POST sequences.

Source: Results, paragraph 2; Figure S19A.

Criterion (claims file): From the cells of fig2b-strong-contributors, the number significant in both PRE and POST over the number significant in PRE, under readings P-a and P-b. Reproduced when within tolerance under one reading.

In [ ]:
print(f2b[["Y", "D", "P", "sig_pre", "both", "pct_pre_also_post"]].to_string(float_format=lambda v: f"{v:.1f}"))
print("expected 73 +- 7.3 %")

**Result: reproduced.** Under Y-a, D-a, P-a, 99 of 134 PRE-significant units are also POST-significant, 73.9% (73 +- 7.3); 72.7% and 71.7% under Y-b and Y-c. Under D-b and under P-b the share is 47.6 to 65.1%.

## rigid-plastic-counts

> Splitting strongly contributing cells at each session's median dPCC gives 105 rigid and 105 plastic cells.

Source: Materials and Methods, Rigid and Plastic Cell Classification.

Criterion (claims file): Classification as in the opening comment (per session and running direction, the median cell dropped for odd counts). Reproduced when both counts are within tolerance.

In [ ]:
print(f2b[["Y", "D", "P", "strong", "rigid", "plastic"]].to_string())
print("expected 105 +- 10.5 each")
MAIN = classify(UNITS[("a", "a")], "a")  # readings Y-a, D-a, P-a for the claims that use the groups
print("groups used below (Y-a, D-a, P-a):", MAIN.group.value_counts().to_dict())
print(MAIN[MAIN.group != ""].groupby(["session", "group"]).size().unstack().to_string())

**Result: partially reproduced, resting on an assumption.** Under the adopted readings Y-a, D-a, P-a there are 121 rigid and 121 plastic units, above 105 +- 10.5. Under D-b the counts are 109, 113 and 108 (Y-a, Y-b, Y-c), within tolerance; under P-b they are 9 to 16. The counts match under one of the two direction readings the claims file names. The group claims are repeated with the D-b groups after figs18.

## figs19c-pcc-change-per-cell

> Testing each place cell's per-event PCC scores between PRE and POST, 2.2% of cells had significantly higher PRE than POST scores and 21.4% significantly higher POST than PRE scores.

Source: Figure S19C legend.

Criterion (claims file): Per the Figure S19C legend, for each place cell (491, per running direction) two one-tailed rank-sum tests of its per-event PCC scores in PRE events against those in POST events, one for each direction, counted at p < 0.05 (the counting threshold of the panel); the denominator is all place cells (the panel's axis, "491 total"). PCC as in figs9b-pcc-post-higher. Reproduced when both percentages are within tolerance under readings Y and D; partially reproduced when the POST > PRE percentage is within tolerance and much larger than the PRE > POST one but a value falls outside tolerance.

In [ ]:
f19 = []
for Y in YS:
    for D in DS:
        u = UNITS[(Y, D)]
        hi_pre = hi_post = 0
        for _, r in u.iterrows():
            a, b = r["_pcc_pre"], r["_pcc_post"]
            if len(a) and len(b):
                hi_pre += stats.ranksums(a, b, alternative="greater").pvalue < 0.05
                hi_post += stats.ranksums(b, a, alternative="greater").pvalue < 0.05
        f19.append(dict(Y=Y, D=D, units=len(u), pct_pre_gt_post=100 * hi_pre / len(u), pct_post_gt_pre=100 * hi_post / len(u)))
print(pd.DataFrame(f19).to_string(float_format=lambda v: f"{v:.2f}"))
print("expected 2.2 +- 0.22 % and 21.4 +- 2.14 %")

**Result: not reproduced.** Under Y-a, D-a, 5.66% of the 565 place units have significantly higher PRE than POST PCC and 14.69% higher POST than PRE, against 2.2 +- 0.22 and 21.4 +- 2.14%; over the six readings the values are 3.72 to 6.37% and 13.45 to 15.93%. POST > PRE is more common than PRE > POST under every reading, but neither value is within tolerance.

In [ ]:
# Per-unit spatial measures and laps, and per-cell offline measures, shared by the
# rigid/plastic and low/high rate claims.
SES = {s["name"]: s for s in SESSIONS}
RESD = {r["name"]: r for r in RES}


def spike_samples(s, c):
    """Index of the position sample of each spike on the maze (within one sample)."""
    t, dt = s["pos_t"], s["pos_dt"]
    sp = s["spikes"][c]
    sp = sp[(sp >= t[0]) & (sp <= t[-1])]
    k = np.clip(np.searchsorted(t, sp), 1, len(t) - 1)
    k = np.where(np.abs(t[k - 1] - sp) < np.abs(t[k] - sp), k - 1, k)
    return k[np.abs(t[k] - sp) < dt]


def laps(s, d):
    """Laps: contiguous stretches of linearised position (the stem, or the circle outside
    the reward area) crossing at least half the track in direction d [S]."""
    x = s["x1d"]
    ok = ~np.isnan(x)
    L = np.nanmax(x)
    out = []
    for a, b in hc11._runs(ok):
        seg = x[a:b]
        net = seg[-1] - seg[0]
        span = seg.max() - seg.min()
        if span < 0.5 * L:
            continue
        if s["circular"] or (d == "right" and net > 0) or (d == "left" and net < 0):
            out.append((a, b))
    return out


def map_from(s, c, mask, nb, edges):
    x = s["x1d"]
    xb = np.clip(np.digitize(x, edges) - 1, 0, nb - 1)
    occ = np.bincount(xb[mask], minlength=nb) * s["pos_dt"]
    k = spike_samples(s, c)
    k = k[mask[k]]
    cnt = np.bincount(xb[k], minlength=nb).astype(float)
    return hc11.smooth(cnt) / np.maximum(hc11.smooth(occ), 1e-9), occ


urows = []
for s, r in zip(SESSIONS, RES):
    x = s["x1d"]
    for d, pm in r["pm"].items():
        edges = pm["edges"]
        nb = len(edges) - 1
        xb = np.clip(np.digitize(x, edges) - 1, 0, nb - 1)
        m = s["dirs"][d]
        lp = laps(s, d)
        lapid = np.full(len(x), -1)
        for i, (a, b) in enumerate(lp):
            lapid[a:b] = i
        idx_run = np.flatnonzero(m)
        half1 = np.zeros(len(x), bool)
        half1[idx_run[: len(idx_run) // 2]] = True
        flip = np.nanmedian(s["vel1d"][m]) < 0  # run starts at the high end of the coordinate
        odd = (lapid >= 0) & (lapid % 2 == 0)
        even = (lapid >= 0) & (lapid % 2 == 1)
        for c in s["pyr"]:
            if not pm["fields"][c]:
                continue
            rate, occ = pm["maps"][c], pm["occ"]
            infield = np.zeros(nb, bool)
            for a, b in pm["fields"][c]:
                infield[a:b] = True
            rin = np.sum(rate[infield] * occ[infield]) / max(occ[infield].sum(), 1e-9)
            rout = np.sum(rate[~infield] * occ[~infield]) / max(occ[~infield].sum(), 1e-9)
            ips, ipsec = hc11.spatial_info(rate, occ)
            m1, _ = map_from(s, c, m & half1, nb, edges)
            m2, _ = map_from(s, c, m & ~half1, nb, edges)
            mo, _ = map_from(s, c, m & odd, nb, edges)
            me, _ = map_from(s, c, m & even, nb, edges)
            # per lap
            k = spike_samples(s, c)
            k = k[m[k] & (lapid[k] >= 0)]
            nsp = np.bincount(lapid[k], minlength=len(lp))
            nin = np.bincount(lapid[k][infield[xb[k]]], minlength=len(lp))
            T = np.bincount(lapid[m & (lapid >= 0)], minlength=len(lp)) * s["pos_dt"]
            Tin = np.bincount(lapid[m & (lapid >= 0) & infield[xb]], minlength=len(lp)) * s["pos_dt"]
            lap_ok = (nsp >= 5) & (T > 0)
            urows.append(dict(
                session=s["name"], animal=s["animal"], cell=c, dir=d, shank=c // 100,
                inout=rin / rout if rout > 0 else np.nan, info_spike=ips, info_sec=ipsec,
                coherence=hc11.coherence(pm["raw"][c]),
                stab_half=np.corrcoef(m1, m2)[0, 1], stab_oddeven=np.corrcoef(mo, me)[0, 1],
                nfields=len(pm["fields"][c]),
                peak_frac=(1 - (np.argmax(rate) + 0.5) / nb) if flip else (np.argmax(rate) + 0.5) / nb,
                n_laps=len(lp), laps_ok=int(lap_ok.sum()),
                _lap=np.flatnonzero(lap_ok), _nsp=nsp[lap_ok], _nin=nin[lap_ok], _T=T[lap_ok], _Tin=Tin[lap_ok],
            ))
SP = pd.DataFrame(urows)


def ripples_in(s, r, iv):
    rp = r["ripples"]
    mid = (rp.start.values + rp.stop.values) / 2
    return rp[hc11.in_intervals(mid, iv)][["start", "stop"]].values


def offline_measures(Y="a"):
    rows = []
    for s, r in zip(SESSIONS, RES):
        pre, post = hc11.yoke(s, Y)
        run = hc11.mask_intervals(s["pos_t"], s["running"], s["pos_dt"])
        for c in s["pyr"]:
            sp = s["spikes"][c]
            row = dict(session=s["name"], cell=c)
            for name, iv in (("pre", pre), ("post", post)):
                rip = ripples_in(s, r, iv)
                T, Tr = hc11.total(iv), float(np.sum(rip[:, 1] - rip[:, 0]))
                n_off = hc11.count_in(sp, iv)
                lo, hi = np.searchsorted(sp, rip[:, 0]), np.searchsorted(sp, rip[:, 1])
                per = hi - lo
                row[f"rate_{name}"] = n_off / T
                row[f"riprate_{name}"] = per.sum() / Tr
                row[f"gain_{name}"] = (per.sum() / Tr) / (n_off / T) if n_off else np.nan
                row[f"part_{name}"] = 100 * np.mean(per > 0)
                # within-ripple ISIs: between spikes of one event; variant, in concatenated ripple time
                isi_in = np.concatenate([np.diff(sp[a:b]) for a, b in zip(lo, hi) if b - a > 1] or [np.zeros(0)])
                row[f"ripburst_{name}"] = 100 * np.mean(isi_in < 0.006) if len(isi_in) else np.nan
                inside = hc11.in_intervals(sp, rip)
                isi_cat = np.diff(sp)[inside[1:] & inside[:-1]]
                row[f"ripburst_cat_{name}"] = 100 * np.mean(isi_cat < 0.006) if len(isi_cat) else np.nan
                row[f"n_isi_{name}"] = len(isi_in)
            both = np.vstack([pre, post])
            row["rate_sess_off"] = hc11.count_in(sp, both) / hc11.total(both)
            row["rate_sess_all"] = np.sum(sp <= s["duration"]) / s["duration"]  # spikes past sessDuration left out
            row["rate_maze_all"] = hc11.count_in(sp, [s["mz"]]) / (s["mz"][1] - s["mz"][0])
            row["rate_maze_run"] = hc11.count_in(sp, run) / hc11.total(run)
            rows.append(row)
    o = pd.DataFrame(rows)
    o["dgain"] = o.gain_post - o.gain_pre
    o["dripburst"] = o.ripburst_post - o.ripburst_pre
    o["dripburst_cat"] = o.ripburst_cat_post - o.ripburst_cat_pre
    return o


OFF = offline_measures("a")
G = MAIN[["session", "cell", "dir", "group"]]
SPG = SP.merge(G, on=["session", "cell", "dir"])
OFFG = MAIN[MAIN.group != ""][["session", "cell", "dir", "group"]].merge(OFF, on=["session", "cell"])
print("units with spatial measures:", len(SP), "; groups:", SPG.group.value_counts().to_dict())
print("laps per session and direction:", SP.groupby(["session", "dir"]).n_laps.first().to_dict())


def compare(df, col, a="plastic", b="rigid", alt="two-sided"):
    x, y = df[df.group == a][col].dropna(), df[df.group == b][col].dropna()
    return dict(measure=col, n_plastic=len(x), n_rigid=len(y), median_plastic=x.median(), median_rigid=y.median(),
                mean_plastic=x.mean(), mean_rigid=y.mean(), p_ranksum=float(stats.ranksums(x, y).pvalue))

lp_tab = SP.groupby(["session", "dir"]).n_laps.first().reset_index()
lp_tab["run_min"] = [SES[s_]["dirs"][d_].sum() * SES[s_]["pos_dt"] / 60 for s_, d_ in zip(lp_tab.session, lp_tab.dir)]
lp_tab["laps_per_run_min"] = lp_tab.n_laps / lp_tab.run_min
print("laps against running time:")
print(lp_tab.to_string(float_format=lambda v: f"{v:.2f}"))


## figs2-rigid-plastic-along-track

> Rigid and plastic cells' place fields are similarly distributed along the track, both enriched near the start and goal ends.

Source: Results, paragraph 2; Figure S2 legend.

Criterion (claims file): The paper shows this only as rate maps and gives no test, so the test is open. Measure, each cell's peak place-field position (per running direction) as a fraction of track length from the start of the run. Readings (a) a two-sample Kolmogorov-Smirnov test of rigid against plastic positions; (b) a rank-sum test of the distance of each peak from the nearer track end. Reproduced when neither reading finds a difference at p < 0.05; the fractions of fields in the outer fifths of the track are reported for each group.

In [ ]:
g = SPG[SPG.group != ""].copy()
g["end_dist"] = np.minimum(g.peak_frac, 1 - g.peak_frac)
a, b = g[g.group == "rigid"], g[g.group == "plastic"]
print(f"rigid n={len(a)}, plastic n={len(b)}")
print(f"(a) KS test of peak positions: D = {stats.ks_2samp(a.peak_frac, b.peak_frac).statistic:.3f}, "
      f"p = {stats.ks_2samp(a.peak_frac, b.peak_frac).pvalue:.3g}")
print(f"(b) rank-sum of distance to the nearer end: medians rigid {a.end_dist.median():.3f}, plastic {b.end_dist.median():.3f}, "
      f"p = {stats.ranksums(a.end_dist, b.end_dist).pvalue:.3g}")
for nm, gg in (("rigid", a), ("plastic", b)):
    print(f"{nm}: fraction of peaks in the outer fifths {np.mean((gg.peak_frac < 0.2) | (gg.peak_frac > 0.8)):.2f} "
          f"(uniform 0.40); in the middle fifth {np.mean((gg.peak_frac >= 0.4) & (gg.peak_frac < 0.6)):.2f}")
print("all place units, fraction in outer fifths:", np.mean((SP.peak_frac < 0.2) | (SP.peak_frac > 0.8)).round(2))
print("peak position as a fraction of track length from the start of the run (the coordinate is flipped where runs go "
      "toward lower values)")

**Result: reproduced, resting on an assumption.** Neither test finds a difference between 121 rigid and 121 plastic units: Kolmogorov-Smirnov D = 0.116, p = 0.39; rank-sum of distance to the nearer end p = 0.22. Both groups are enriched at the ends, 0.64 (rigid) and 0.71 (plastic) of peaks in the outer fifths against 0.40 for a uniform spread.

## fig3a-plastic-more-specific

> On the novel maze plastic cells have higher place specificity than rigid cells and fewer place fields.

Source: Figure 3A; Results, paragraph 3; Figure S19E.

Criterion (claims file): The Figure 3A panels are not available here; Figure S19E, which the legend says to compare to Figure 3A, shows six measures, within/outside place-field rate ratio, information per spike, information per second, spatial coherence, place-field stability and number of place fields, and marks every one of the six as significant, plastic higher on the first five and lower on the number of fields. Measures as in reference 32 (Mizuseki et al. 2012) per the Methods; the stability split is not given here (readings (a) first against second half of the maze epoch; (b) odd against even laps). Rank-sum test, rigid against plastic (Figure 3A legend). Reproduced when plastic cells are higher on all five specificity measures and have fewer place fields, each at p < 0.05, under one stability reading; partially reproduced when the direction holds for all six but fewer reach p < 0.05.

In [ ]:
cols = ["inout", "info_spike", "info_sec", "coherence", "stab_half", "stab_oddeven", "nfields"]
t3a = pd.DataFrame([compare(SPG, c) for c in cols])
t3a["plastic_higher"] = t3a.median_plastic > t3a.median_rigid
t3a["plastic_higher_mean"] = t3a.mean_plastic > t3a.mean_rigid
print(t3a.to_string(float_format=lambda v: f"{v:.4g}"))
print("geometric mean number of fields, plastic %.3f rigid %.3f" % tuple(
    np.exp(np.log(SPG[SPG.group == gname].nfields).mean()) for gname in ("plastic", "rigid")))
print("by animal, median in/out ratio:")
print(SPG[SPG.group != ""].groupby(["animal", "group"]).inout.median().unstack().round(2).to_string())

**Result: partially reproduced, resting on an assumption.** Plastic units are higher on all five specificity measures, each at p < 0.05: in/out ratio (median 8.98 against 5.43, p = 2.2e-05), information per spike (p = 5.9e-05), information per second (p = 0.0047), coherence (p = 0.0049) and stability (first against second half p = 0.013; odd against even laps p = 0.001). They have fewer place fields on average (mean 1.347 against 1.397, geometric mean 1.242 against 1.287), but not significantly (p = 0.51).

## fig3b-plastic-specificity-increases

> Plastic but not rigid cells steadily increased their spatial specificity during maze running.

Source: Figure 3B; Results, paragraph 3; Figure S19F.

Criterion (claims file): Per lap, each cell's percentage of spikes inside its place field(s) (the measure of Figure S19F, the legend's "infield versus outside field firing ratio"), using only laps with at least 5 spikes from the cell (Figure S13 legend); per cell, the Pearson correlation of this value with lap number. Tests per the Figure S13C legend, a signed-rank test of each group's correlations against zero and a rank-sum test between groups. Reproduced when plastic cells' correlations are above zero (p < 0.05), rigid cells' are not, and plastic exceeds rigid (p < 0.05). The within-field rate gain of Figure S13C is shown as a variant.

In [ ]:
def lap_corr(row, kind):
    lap, nsp, nin, T, Tin = row["_lap"], row["_nsp"], row["_nin"], row["_T"], row["_Tin"]
    if len(lap) < 3:
        return np.nan
    with np.errstate(divide="ignore", invalid="ignore"):
        v = {"pct_in": 100 * nin / nsp, "rate": nsp / T, "rate_in": np.where(Tin > 0, nin / Tin, np.nan),
             "rate_out": np.where(T - Tin > 0, (nsp - nin) / (T - Tin), np.nan),
             "gain": np.where(Tin > 0, (nin / Tin) / (nsp / T), np.nan)}[kind]
    ok = ~np.isnan(v)
    if ok.sum() < 3 or np.std(v[ok]) == 0:
        return np.nan
    return stats.pearsonr(lap[ok], v[ok])[0]


for kind in ("pct_in", "gain", "rate", "rate_in", "rate_out"):
    SPG[f"r_{kind}"] = SPG.apply(lambda r: lap_corr(r, kind), axis=1)


def lap_summary(kind):
    out = {}
    for gname in ("rigid", "plastic"):
        v = SPG[SPG.group == gname][f"r_{kind}"].dropna()
        out[gname] = (len(v), v.mean(), hc11.wilcoxon_1s(v, "greater"), hc11.wilcoxon_1s(v, "less"))
        print(f"{kind:8s} {gname:7s}: n = {len(v)}, mean r = {v.mean():+.3f}, median {v.median():+.3f}, "
              f"signed-rank p(>0) = {out[gname][2]:.3g}, p(<0) = {out[gname][3]:.3g}")
    x, y = SPG[SPG.group == "plastic"][f"r_{kind}"].dropna(), SPG[SPG.group == "rigid"][f"r_{kind}"].dropna()
    print(f"{kind:8s} plastic > rigid rank-sum p = {stats.ranksums(x, y, alternative='greater').pvalue:.3g}")
    return out


_ = lap_summary("pct_in")
print("variant, within-field gain (Figure S13C):")
_ = lap_summary("gain")
gg = SPG[SPG.group != ""]
print(f"dropped: units with fewer than 3 laps of >= 5 spikes: {gg.r_pct_in.isna().sum()} of {len(gg)}")

**Result: not reproduced, resting on an assumption.** The per-lap percentage of spikes in field rises with lap number in both groups by similar amounts: plastic mean r = +0.052 (signed-rank p = 0.10 for r > 0), rigid +0.046 (p = 0.15), plastic against rigid p = 0.48 (103 units each). The within-field gain variant goes the other way (rigid p = 0.038, plastic p = 0.20).

## figs13-outfield-rate-falls

> Neither group's overall firing rate changed significantly over laps, and neither group's within-field firing rate increased significantly; plastic cells' increase in specificity came from a fall in their out-of-field firing rate, which rigid cells did not show.

Source: Figure S13A, S13D and S13E.

Criterion (claims file): Per cell, the Pearson correlation of lap rate (overall, within field, outside field) with lap number, laps with at least 5 spikes, running above 5 cm/s. Signed-rank test of each group's correlations against zero (Figure S13C legend). Reproduced when the outside-field correlations of plastic cells are below zero (p < 0.05) and those of rigid cells are not, neither group's overall-rate correlations differ from zero at p < 0.05, and neither group's within-field correlations are above zero at p < 0.05.

In [ ]:
for kind in ("rate", "rate_in", "rate_out"):
    _ = lap_summary(kind)
    for gname in ("rigid", "plastic"):
        v = SPG[SPG.group == gname][f"r_{kind}"].dropna()
        print(f"   two-sided signed-rank p, {gname}: {hc11.wilcoxon_1s(v, 'two-sided'):.3g}")

**Result: reproduced, resting on an assumption.** Plastic units' out-of-field rate falls over laps (mean r = -0.087, signed-rank p = 0.015 for r < 0) and rigid units' does not (p = 0.118). Neither group's overall rate changes (two-sided p = 0.94 rigid, 0.10 plastic), and neither group's within-field rate rises (p = 0.33 and 0.80 for r > 0). The outcome rests on the D-a groups: with the D-b groups under Y-b the plastic overall rate also falls (two-sided p = 0.047, cell under "Rigid and plastic claims under the other group readings"), which the claim excludes.

## fig4a-rigid-fire-faster

> Rigid cells fire faster than plastic cells in offline states (session-wide median 0.76 against 0.54 Hz; PRE 0.762 against 0.479 Hz; POST 0.753 against 0.472 Hz), but not during maze running (1.09 against 1.13 Hz).

Source: Figure 4A; Results, paragraph 4; Figure S14, firing rate row (values read from the panel titles).

Criterion (claims file): Firing rate as in Figure S14, spikes per second in offline states of each epoch, and during running on the maze for MAZE (the measure for MAZE is not stated; readings (a) all MAZE time; (b) running above 5 cm/s). "Session-wide" is not defined; readings (a) offline time of PRE and POST together; (b) the whole recording. Rank-sum tests. Reproduced when the session-wide, PRE and POST differences are significant (p < 0.05) with rigid higher, the MAZE difference is not (rank-sum p >= 0.05), and the medians are within tolerance under one reading; partially reproduced when the tests hold and the medians do not.

In [ ]:
for Y in YS:
    o = offline_measures(Y) if Y != "a" else OFF
    og = MAIN[MAIN.group != ""][["session", "cell", "dir", "group"]].merge(o, on=["session", "cell"])
    t = pd.DataFrame([compare(og, c) for c in ("rate_sess_off", "rate_sess_all", "rate_pre", "rate_post",
                                                "rate_maze_all", "rate_maze_run")])
    print(f"reading Y-{Y}")
    print(t.to_string(float_format=lambda v: f"{v:.4g}"))
print("expected medians rigid/plastic: session 0.76/0.54, PRE 0.762/0.479, POST 0.753/0.472 Hz (10%)")
print("units counted once per running direction; distinct cells:",
      OFFG.drop_duplicates(["session", "cell", "group"]).group.value_counts().to_dict())
print("by animal, median session-wide offline rate:")
print(OFFG.groupby([OFFG.session.str.split("_").str[0], "group"]).rate_sess_off.median().unstack().round(3).to_string())

**Result: not reproduced, resting on an assumption.** Rigid units fire faster than plastic units in PRE (median 0.697 against 0.461 Hz, p = 0.0033; both within tolerance), but not significantly session-wide (0.671 against 0.588 Hz, p = 0.098 offline; 0.686 against 0.627, p = 0.14 over the whole recording) or in POST (0.609 against 0.530 Hz, p = 0.31). On the maze there is no difference (p = 0.40 all MAZE time, 0.61 running). Rigid medians are higher in every comparison off the maze, and the Y readings change no test.

## figs14-plastic-ripple-gain

> Plastic cells have a higher within-ripple firing rate gain than rigid cells (POST median 5.75 against 4.52) and a larger PRE to POST increase in gain (0.81 against 0.14).

Source: Figure 4A; Results, paragraph 4; Figure S14, within-ripple firing rate gain row (values read from the panel titles).

Criterion (claims file): Gain as in the Figure S14 legend, within-ripple rate over offline rate, per epoch. Rank-sum tests, plastic against rigid. The rigid change median is near zero, so its tolerance is set at 0.1, about 10% of the gap between the two groups' changes. Reproduced when plastic exceeds rigid for POST gain and for the change at p < 0.05 and the four medians are within tolerance; partially reproduced when the tests hold and a median does not.

In [ ]:
t = pd.DataFrame([compare(OFFG, c) for c in ("gain_pre", "gain_post", "dgain", "riprate_pre", "riprate_post",
                                             "part_pre", "part_post")])
print(t.to_string(float_format=lambda v: f"{v:.4g}"))
for c in ("gain_post", "dgain"):
    x, y = OFFG[OFFG.group == "plastic"][c].dropna(), OFFG[OFFG.group == "rigid"][c].dropna()
    print(f"{c}: plastic > rigid one-sided rank-sum p = {stats.ranksums(x, y, alternative='greater').pvalue:.3g}")
print("expected: POST gain plastic 5.75 +- 0.575, rigid 4.52 +- 0.452; change plastic 0.81 +- 0.081, rigid 0.14 +- 0.1")
print("gain range:", np.nanmin(OFFG[["gain_pre", "gain_post"]].values).round(2), "to",
      np.nanmax(OFFG[["gain_pre", "gain_post"]].values).round(2))

**Result: not reproduced, resting on an assumption.** The POST gain is higher in plastic units (median 5.01 against 4.53, one-sided rank-sum p = 0.029, two-sided 0.057), but the PRE to POST change is not significantly larger (0.78 against 0.55, one-sided p = 0.074). The rigid POST median is within tolerance (4.52 +- 0.452); the plastic POST median (5.75 +- 0.575) and both change medians (0.81 +- 0.081, 0.14 +- 0.1) are not.

## figs14-plastic-ripple-bursting

> Plastic cells burst more within ripples than rigid cells in POST (median 27.3% against 23.3% of ISIs under 6 ms) and increase their within-ripple bursting more from PRE to POST (2.81 against 1.13 percentage points).

Source: Figure S14, within-ripple burstiness row (values read from the panel titles).

Criterion (claims file): Within-ripple burstiness as the percentage of within-ripple ISIs under 6 ms. Whether ISIs spanning two ripple events count is open; the reading kept is ISIs between spikes of one event (standard), with all ISIs inside concatenated ripple time as a variant. Rank-sum tests. Reproduced when both tests hold with plastic higher and the medians are within tolerance; partially reproduced when the tests hold and a median does not.

In [ ]:
t = pd.DataFrame([compare(OFFG, c) for c in ("ripburst_pre", "ripburst_post", "dripburst",
                                             "ripburst_cat_post", "dripburst_cat")])
print(t.to_string(float_format=lambda v: f"{v:.4g}"))
for c in ("ripburst_post", "dripburst", "ripburst_cat_post", "dripburst_cat"):
    x, y = OFFG[OFFG.group == "plastic"][c].dropna(), OFFG[OFFG.group == "rigid"][c].dropna()
    print(f"{c}: plastic > rigid one-sided rank-sum p = {stats.ranksums(x, y, alternative='greater').pvalue:.3g}")
print("expected: POST plastic 27.3 +- 2.73 %, rigid 23.3 +- 2.33 %; change plastic 2.81 +- 0.281, rigid 1.13 +- 0.113 points")
print("units without within-ripple ISIs (PRE or POST):", int(OFFG[["ripburst_pre", "ripburst_post"]].isna().any(axis=1).sum()))

**Result: not reproduced, resting on an assumption.** In POST plastic units burst more within ripples (median 28.3 against 23.5%, one-sided p = 0.0035), both medians within tolerance; with ISIs over concatenated ripple time p = 0.0031. The PRE to POST change is not larger in plastic units (median 0.43 against 0.77 points, p = 0.53), against 2.81 and 1.13 expected.

In [ ]:
# Pair-wise measures: coactivity (100 ms bins) and temporal bias (CCG +-150 ms) for all
# place cells (cells with a place field in any direction), pairs on different shanks [M].
def binned(sp, iv, width=0.1):
    """Counts in non-overlapping bins laid within each interval (partial bins dropped)."""
    out = []
    for a, b in iv:
        n = int((b - a) // width)
        if n < 1:
            continue
        e = a + width * np.arange(n + 1)
        out.append(np.diff(np.searchsorted(sp, e)))
    return np.concatenate(out) if out else np.zeros(0)


def ccg_counts(a, b, iv, half=0.15):
    """Spikes of b within half s after and before each spike of a, for spikes of a in iv."""
    a = a[hc11.in_intervals(a, iv)]
    after = np.searchsorted(b, a + half, side="right") - np.searchsorted(b, a, side="right")
    before = np.searchsorted(b, a, side="left") - np.searchsorted(b, a - half, side="left")
    return after.sum(), before.sum()


PAIRS = []
CELLGROUP = {}
for (sname, c), g in MAIN.groupby(["session", "cell"]):
    labels = set(g.group) - {""}
    CELLGROUP[(sname, c)] = labels.pop() if len(labels) == 1 else ("mixed" if labels else "")
for s, r in zip(SESSIONS, RES):
    pre, post = hc11.yoke(s, "a")
    run = hc11.mask_intervals(s["pos_t"], s["running"], s["pos_dt"])
    cells = sorted({c for d, pm in r["pm"].items() for c in s["pyr"] if pm["fields"][c]})
    B = {nm: np.array([binned(s["spikes"][c], iv) for c in cells]) for nm, iv in (("PRE", pre), ("POST", post), ("RUN", run))}
    C = {nm: np.corrcoef(b) for nm, b in B.items()}
    for i, a in enumerate(cells):
        for j in range(i + 1, len(cells)):
            b = cells[j]
            if a // 100 == b // 100:
                continue
            row = dict(session=s["name"], a=a, b=b, ga=CELLGROUP[(s["name"], a)], gb=CELLGROUP[(s["name"], b)])
            for nm, iv in (("PRE", pre), ("POST", post), ("RUN", run)):
                row[f"co_{nm}"] = C[nm][i, j]
                af, bf = ccg_counts(s["spikes"][a], s["spikes"][b], iv)
                row[f"n_{nm}"] = af + bf
                # reference a fires after comparison b = b spikes before a spikes
                row[f"bias_{nm}"] = (bf - af) / (af + bf) if af + bf else np.nan
            PAIRS.append(row)
PAIRS = pd.DataFrame(PAIRS)
PAIRS["dco"] = PAIRS.co_POST - PAIRS.co_PRE
print("pairs on different shanks:", len(PAIRS), "; per session:", PAIRS.groupby("session").size().to_dict())
print("coactivity ranges:", {k: (round(PAIRS[k].min(), 3), round(PAIRS[k].max(), 3)) for k in ("co_PRE", "co_RUN", "co_POST")},
      "; pairs with an undefined value:", int(PAIRS[["co_PRE", "co_RUN", "co_POST"]].isna().any(axis=1).sum()))

## figs16ab-plastic-synchrony-increase

> The PRE to POST increase in ripple-triggered firing is present in plastic but not rigid cells, and the PRE to POST increase in pair-wise coactivity is larger among plastic cells.

Source: Figure S16A and S16B; Figure 4A and Figure S19G (change in pair-wise coactivity).

Criterion (claims file): Ripple-triggered rate from -50 to 100 ms around event onset (the panel's axis), per cell, PRE and POST. The legend gives no test for panel A; readings (a) a signed-rank test of each cell's POST - PRE mean rate in 0 to 100 ms; (b) the same at the peak bin. Coactivity as in the opening comment. Figure S16B shows a significant PRE to POST increase within both rigid and plastic pairs, larger in plastic pairs, with no test between groups; Figure 4A (and Figure S19G, which shows it with p < 0.023) compares the per-cell POST - PRE change in pair-wise coactivity between rigid and plastic cells by rank-sum test. How a cell's change is formed from its pairs is not stated; readings (c) the mean change over the cell's pairs with all other place cells on other shanks; (d) the mean over its pairs within its own group. Reproduced when plastic cells' rate increase is significant (p < 0.05) and rigid cells' is not under one reading of A, and plastic cells' coactivity change exceeds rigid cells' (rank-sum, p < 0.05) under one reading of the per-cell change.

In [ ]:
def ripple_psth(s, r, c, iv, edges):
    rip = ripples_in(s, r, iv)
    sp = s["spikes"][c]
    h = np.zeros(len(edges) - 1)
    for t0 in rip[:, 0]:
        lo, hi = np.searchsorted(sp, t0 + edges[0]), np.searchsorted(sp, t0 + edges[-1])
        h += np.histogram(sp[lo:hi] - t0, edges)[0]
    return h / max(len(rip), 1) / np.diff(edges)


edges = np.arange(-0.05, 0.1001, 0.01)
ps = []
for _, row in OFFG.iterrows():
    s, r = SES[row.session], RESD[row.session]
    pre, post = hc11.yoke(s, "a")
    a = ripple_psth(s, r, row.cell, pre, edges)
    b = ripple_psth(s, r, row.cell, post, edges)
    win = edges[:-1] >= 0
    ps.append(dict(group=row.group, d_mean=b[win].mean() - a[win].mean(), d_peak=b[win].max() - a[win].max()))
ps = pd.DataFrame(ps)
for gname in ("rigid", "plastic"):
    g = ps[ps.group == gname]
    print(f"{gname}: n = {len(g)}; (a) POST - PRE mean rate 0-100 ms: median {g.d_mean.median():+.3f} Hz, "
          f"signed-rank p(>0) = {hc11.wilcoxon_1s(g.d_mean):.3g}; (b) peak bin: median {g.d_peak.median():+.3f} Hz, "
          f"p(>0) = {hc11.wilcoxon_1s(g.d_peak):.3g}")
# per-cell change in coactivity: (c) over pairs with all other place cells, (d) over pairs within its group
long = pd.concat([PAIRS.rename(columns={"a": "cell", "gb": "other"}).assign(own=PAIRS.ga),
                  PAIRS.rename(columns={"b": "cell", "ga": "other"}).assign(own=PAIRS.gb)])
for reading, sub in (("c all partners", long), ("d own-group partners", long[long.other == long.own])):
    pc = sub.groupby(["session", "cell", "own"]).dco.mean().reset_index()
    x, y = pc[pc.own == "plastic"].dco, pc[pc.own == "rigid"].dco
    print(f"({reading}) per-cell POST - PRE coactivity: plastic n = {len(x)} median {x.median():+.4f}, rigid n = {len(y)} "
          f"median {y.median():+.4f}; plastic > rigid rank-sum p = {stats.ranksums(x, y, alternative='greater').pvalue:.3g}")
for gname in ("rigid", "plastic"):
    g = PAIRS[(PAIRS.ga == gname) & (PAIRS.gb == gname)].dco.dropna()
    print(f"{gname}-{gname} pairs: n = {len(g)}, mean POST - PRE coactivity {g.mean():+.4f}, signed-rank p(>0) = {hc11.wilcoxon_1s(g):.3g}")
print("cells labelled rigid in one direction and plastic in the other (left out of cell-level groups):",
      sum(v == "mixed" for v in CELLGROUP.values()))

**Result: reproduced, resting on an assumption.** The ripple-triggered rate rises from PRE to POST in plastic units (median +0.347 Hz over 0 to 100 ms, signed-rank p = 0.0021; peak bin +0.479 Hz, p = 0.0012) and not in rigid units (p = 0.73 and 0.63). Plastic cells' per-cell coactivity change exceeds rigid cells' with all partners (median +0.0078 against +0.0033, one-sided p = 0.035, two-sided 0.07) and with own-group partners (+0.0115 against +0.0002, one-sided p = 3.5e-07). The outcome rests on reading (d), as reading (c) is not significant two-sided.

## figs17-coactivity-preservation

> Pair-wise coactivity is weakly preserved from PRE to RUN (r = 0.045), more strongly from RUN to POST (r = 0.16), and most strongly from PRE to POST (r = 0.34).

Source: Figure S17A and S17C (r values read from the panels).

Criterion (claims file): Coactivity as in the opening comment, all place-cell pairs on different shanks; Pearson correlation across pairs between epochs; differences of correlations by Fisher's Z test (as two independent correlations, the test's usual form; a test for dependent correlations is the other reading). Reproduced when both differences are significant (p < 0.05) in the stated direction and the values are within tolerance; partially reproduced when the order RUN-PRE < RUN-POST < PRE-POST holds but values fall outside tolerance.

In [ ]:
def fisher_ind(r1, r2, n1, n2):
    z = (np.arctanh(r1) - np.arctanh(r2)) / np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    return z, 2 * stats.norm.sf(abs(z))


def steiger(r1, r2, r12, n):
    """Correlations r1 = corr(X, Y), r2 = corr(X, Z) sharing X; r12 = corr(Y, Z) (Meng et al. 1992)."""
    rm2 = (r1**2 + r2**2) / 2
    f = min((1 - r12) / (2 * (1 - rm2)), 1)
    h = (1 - f * rm2) / (1 - rm2)
    z = (np.arctanh(r1) - np.arctanh(r2)) * np.sqrt((n - 3) / (2 * (1 - r12) * h))
    return z, 2 * stats.norm.sf(abs(z))


def preservation(df):
    d = df[["co_PRE", "co_RUN", "co_POST"]].dropna()
    n = len(d)
    r = {k: stats.pearsonr(d[a], d[b])[0] for k, (a, b) in
         {"PRE-RUN": ("co_PRE", "co_RUN"), "RUN-POST": ("co_RUN", "co_POST"), "PRE-POST": ("co_PRE", "co_POST")}.items()}
    return n, r


n, r = preservation(PAIRS)
print(f"all place-cell pairs, n = {n}: " + ", ".join(f"r({k}) = {v:.3f}" for k, v in r.items()))
print("expected 0.045 +- 0.0045, 0.16 +- 0.016, 0.34 +- 0.034")
d1, d2 = r["RUN-POST"] - r["PRE-RUN"], r["PRE-POST"] - r["RUN-POST"]
print(f"corr(RUN,POST) - corr(RUN,PRE) = {d1:.3f} (expected 0.11 +- 0.011): independent Fisher z p = {fisher_ind(r['RUN-POST'], r['PRE-RUN'], n, n)[1]:.3g}, "
      f"dependent (Steiger) p = {steiger(r['RUN-POST'], r['PRE-RUN'], r['PRE-POST'], n)[1]:.3g}")
print(f"corr(PRE,POST) - corr(RUN,POST) = {d2:.3f} (expected 0.18 +- 0.018): independent Fisher z p = {fisher_ind(r['PRE-POST'], r['RUN-POST'], n, n)[1]:.3g}, "
      f"dependent (Steiger) p = {steiger(r['PRE-POST'], r['RUN-POST'], r['PRE-RUN'], n)[1]:.3g}")
print("per session:")
for sname, g in PAIRS.groupby("session"):
    n_, r_ = preservation(g)
    print(f"  {sname}: n = {n_}, " + ", ".join(f"{k} {v:.3f}" for k, v in r_.items()))

**Result: partially reproduced.** Over 11,010 pairs the order holds, r(PRE, RUN) = 0.054 < r(RUN, POST) = 0.166 < r(PRE, POST) = 0.393, and both differences are significant (0.112, p = 3.1e-17, and 0.227, p = 1.2e-75, independent Fisher z; the dependent test agrees). r(RUN, POST) and the first difference are within tolerance; r(PRE, RUN), r(PRE, POST) and the second difference (0.18 +- 0.018) are not.

## figs17d-rigid-plastic-coactivity

> Rigid cell pairs preserve PRE coactivity into RUN (r = 0.139) and into POST (r = 0.31) more than plastic pairs (-0.02 and 0.09), while plastic pairs show higher RUN to POST preservation (0.36 against 0.11).

Source: Figure S17D (r values read from the panels); Results, paragraph 4.

Criterion (claims file): As figs17-coactivity-preservation within rigid-rigid and plastic-plastic pairs; groups compared by Fisher's Z test (Figure S17D legend). The plastic PRE-RUN r is near zero, so its tolerance is 0.05, read as consistent with zero. Reproduced when the three group differences are significant (p < 0.05) in the stated direction; the r values are reported against tolerance, and a mismatch in values with the differences holding is partially reproduced.

In [ ]:
res17 = {}
for gname in ("rigid", "plastic"):
    sub = PAIRS[(PAIRS.ga == gname) & (PAIRS.gb == gname)]
    res17[gname] = preservation(sub)
    print(f"{gname}-{gname} pairs: n = {res17[gname][0]}, " + ", ".join(f"r({k}) = {v:.3f}" for k, v in res17[gname][1].items()))
print("expected rigid 0.139, 0.11, 0.31; plastic -0.02, 0.36, 0.09 (PRE-RUN, RUN-POST, PRE-POST)")
for k, want in (("PRE-RUN", "rigid"), ("PRE-POST", "rigid"), ("RUN-POST", "plastic")):
    (nr, rr), (np_, rp) = res17["rigid"], res17["plastic"]
    z, p = fisher_ind(rr[k], rp[k], nr, np_)
    print(f"{k}: rigid {rr[k]:.3f} vs plastic {rp[k]:.3f}, Fisher z = {z:.2f}, p = {p:.3g}; "
          f"stated direction ({want} higher) holds: {(rr[k] > rp[k]) == (want == 'rigid')}")

**Result: not reproduced, resting on an assumption.** Rigid pairs preserve PRE coactivity into POST more than plastic pairs (0.530 against 0.218, Fisher z p = 8.2e-12), but the other two differences, though in the stated direction, are not significant: PRE-RUN 0.089 against 0.001 (p = 0.10) and RUN-POST 0.176 against 0.256 (p = 0.12).

## figs18-temporal-bias

> Pair-wise temporal biases are not preserved from PRE to RUN (r = -0.009) but are preserved from RUN to POST (r = 0.117) and PRE to POST (r = 0.201); the PRE to POST preservation is carried by rigid cell pairs (r = 0.29) and the RUN to POST preservation by plastic cell pairs (r = 0.26).

Source: Figure S18B and S18C (r values read from the panels); Results, paragraph 4.

Criterion (claims file): Temporal bias and its median r over 1,000 sign randomisations as in the opening comment. The PRE-RUN value is near zero; its tolerance of 0.05 reads it as consistent with zero. Reproduced when the PRE-RUN r is not significant (p > 0.05), RUN-POST and PRE-POST are (p < 0.05), the rigid PRE-POST and plastic RUN-POST r are significant while the plastic PRE-POST (-0.037) and rigid RUN-POST (0.024) are not, and the values are within tolerance; partially reproduced when the significance pattern holds and a value does not.

In [ ]:
rng18 = np.random.default_rng(18)


def bias_similarity(df, e1, e2, n_perm=1000):
    d = df[(df[f"n_{e1}"] >= 40) & (df[f"n_{e2}"] >= 40)][[f"bias_{e1}", f"bias_{e2}"]].dropna().values
    if len(d) < 4:
        return len(d), np.nan, np.nan
    rs, ps = [], []
    for _ in range(n_perm):
        sg = rng18.choice([-1.0, 1.0], len(d))
        r, p = stats.pearsonr(d[:, 0] * sg, d[:, 1] * sg)
        rs.append(r)
        ps.append(p)
    return len(d), float(np.median(rs)), float(np.median(ps))


rows18 = []
for label, sub in (("all", PAIRS), ("rigid", PAIRS[(PAIRS.ga == "rigid") & (PAIRS.gb == "rigid")]),
                   ("plastic", PAIRS[(PAIRS.ga == "plastic") & (PAIRS.gb == "plastic")])):
    for e1, e2 in (("PRE", "RUN"), ("RUN", "POST"), ("PRE", "POST")):
        n_, r_, p_ = bias_similarity(sub, e1, e2)
        rows18.append(dict(pairs=label, comparison=f"{e1}-{e2}", n=n_, median_r=r_, median_p=p_))
print(pd.DataFrame(rows18).to_string(float_format=lambda v: f"{v:.4g}"))
print("expected all: PRE-RUN -0.009 (+-0.05), RUN-POST 0.117 +- 0.0117, PRE-POST 0.201 +- 0.0201; "
      "rigid PRE-POST 0.29 +- 0.029; plastic RUN-POST 0.26 +- 0.026; plastic PRE-POST and rigid RUN-POST not significant")

**Result: not reproduced, resting on an assumption.** Over all pairs, RUN-POST (median r = 0.110) and PRE-POST (0.215) are significant and within tolerance, but PRE-RUN, -0.022, within tolerance of -0.009, is also significant (median p = 0.044, 8,626 pairs), which the criterion excludes. Rigid pairs carry PRE-POST (0.298, p = 1.1e-16) and plastic pairs RUN-POST (0.177, p = 2.3e-04, below 0.26 +- 0.026), while plastic PRE-POST (p = 0.080) and rigid RUN-POST (p = 0.27) are not significant.

## Rigid and plastic claims under the other group readings

The claims above use groups from readings Y-a, D-a, P-a (121 rigid and 121 plastic units). Under D-b the counts match the paper's 105 (rigid-plastic-counts), so each group claim's main tests are repeated here with the D-b groups under Y-a, Y-b and Y-c. The per-cell offline measures are also repeated with the D-a groups counted once per cell (units counted per direction appear twice otherwise). Rank-sum p-values in this table are two-sided.

In [ ]:
# Variant: the rigid/plastic claims rerun with the groups of the other readings.
# Groups D-b (events assigned to one direction by equation 6) under Y-a, Y-b and Y-c, and
# the adopted D-a groups counted once per cell (cells in both groups, "mixed", left out) for
# the per-cell offline measures. Offline measures, PAIRS and the per-lap values are those
# computed above (Y-a); only the group labels change.
rng_gv = np.random.default_rng(72)


def cell_groups(G):
    out = {}
    for (sname, c), g in G.groupby(["session", "cell"]):
        labels = set(g.group) - {""}
        out[(sname, c)] = labels.pop() if len(labels) == 1 else ("mixed" if labels else "")
    return out


def group_tests(G, distinct=False):
    """Main tests of each rigid/plastic claim for the group labels G (session, cell, dir, group)."""
    G = G[["session", "cell", "dir", "group"]]
    spg = SPG.drop(columns="group").merge(G, on=["session", "cell", "dir"])
    gg = G[G.group != ""]
    cg = cell_groups(G)
    if distinct:
        gg = gg.assign(cg=[cg[(s_, c_)] for s_, c_ in zip(gg.session, gg.cell)])
        gg = gg[gg.cg != "mixed"].drop_duplicates(["session", "cell"])
    offg = gg.merge(OFF, on=["session", "cell"])
    R = {}
    rg, pl = spg[spg.group == "rigid"], spg[spg.group == "plastic"]
    ed = lambda v: np.minimum(v, 1 - v)
    R["figs2"] = dict(n_rigid=len(rg), n_plastic=len(pl), ks_p=stats.ks_2samp(rg.peak_frac, pl.peak_frac).pvalue,
                      end_p=stats.ranksums(ed(rg.peak_frac), ed(pl.peak_frac)).pvalue)
    d = {}
    for c in ("inout", "info_spike", "info_sec", "coherence", "stab_half", "nfields"):
        x, y = pl[c].dropna(), rg[c].dropna()
        d[f"{c}_med_pl"], d[f"{c}_med_rg"], d[f"{c}_p"] = x.median(), y.median(), stats.ranksums(x, y).pvalue
    d["nfields_geomean_pl"], d["nfields_geomean_rg"] = np.exp(np.log(pl.nfields).mean()), np.exp(np.log(rg.nfields).mean())
    R["fig3a"] = d
    d = {}
    for kind, alts in (("pct_in", ("greater",)), ("rate", ("two-sided",)), ("rate_in", ("greater",)), ("rate_out", ("less",))):
        for gname, g in (("pl", pl), ("rg", rg)):
            v = g[f"r_{kind}"].dropna()
            d[f"{kind}_{gname}_mean_r"] = v.mean()
            d[f"{kind}_{gname}_p_{alts[0][:4]}"] = hc11.wilcoxon_1s(v, alts[0])
    d["pct_in_pl_gt_rg_p"] = stats.ranksums(pl.r_pct_in.dropna(), rg.r_pct_in.dropna(), alternative="greater").pvalue
    R["fig3b_figs13"] = d
    d = dict(n_rigid=int((offg.group == "rigid").sum()), n_plastic=int((offg.group == "plastic").sum()))
    for c in ("rate_sess_off", "rate_sess_all", "rate_pre", "rate_post", "rate_maze_run", "gain_post", "dgain",
              "ripburst_post", "dripburst"):
        x, y = offg[offg.group == "plastic"][c].dropna(), offg[offg.group == "rigid"][c].dropna()
        d[f"{c}_med_rg"], d[f"{c}_med_pl"], d[f"{c}_p2"] = y.median(), x.median(), stats.ranksums(x, y).pvalue
    R["fig4a_figs14"] = d
    d = {}
    win = edges[:-1] >= 0
    psr = []
    for row in offg.itertuples():
        s, r = SES[row.session], RESD[row.session]
        p0, p1 = hc11.yoke(s, "a")
        a, b = ripple_psth(s, r, row.cell, p0, edges), ripple_psth(s, r, row.cell, p1, edges)
        psr.append((row.group, b[win].mean() - a[win].mean()))
    psr = pd.DataFrame(psr, columns=["group", "d_mean"])
    for gname in ("rigid", "plastic"):
        d[f"psth_{gname}_p"] = hc11.wilcoxon_1s(psr[psr.group == gname].d_mean)
    pr = PAIRS.assign(ga=[cg[(s_, c_)] for s_, c_ in zip(PAIRS.session, PAIRS.a)],
                      gb=[cg[(s_, c_)] for s_, c_ in zip(PAIRS.session, PAIRS.b)])
    lg = pd.concat([pr.rename(columns={"a": "cell", "gb": "other"}).assign(own=pr.ga),
                    pr.rename(columns={"b": "cell", "ga": "other"}).assign(own=pr.gb)])
    for reading, sub in (("c", lg), ("d", lg[lg.other == lg.own])):
        pc = sub.groupby(["session", "cell", "own"]).dco.mean().reset_index()
        x, y = pc[pc.own == "plastic"].dco, pc[pc.own == "rigid"].dco
        d[f"coact_{reading}_p2"] = stats.ranksums(x, y).pvalue
        d[f"coact_{reading}_pl_gt_rg"] = x.median() > y.median()
    R["figs16ab"] = d
    d = {}
    res = {gname: preservation(pr[(pr.ga == gname) & (pr.gb == gname)]) for gname in ("rigid", "plastic")}
    for k in ("PRE-RUN", "RUN-POST", "PRE-POST"):
        (nr, rr), (np_, rp) = res["rigid"], res["plastic"]
        d[f"{k}_rg"], d[f"{k}_pl"], d[f"{k}_p"] = rr[k], rp[k], fisher_ind(rr[k], rp[k], nr, np_)[1]
    R["figs17d"] = d
    d = {}
    for label in ("rigid", "plastic"):
        sub = pr[(pr.ga == label) & (pr.gb == label)]
        for e1, e2 in (("RUN", "POST"), ("PRE", "POST")):
            n_, r_, p_ = bias_similarity(sub, e1, e2)
            d[f"{label}_{e1}-{e2}_r"], d[f"{label}_{e1}-{e2}_p"] = r_, p_
    R["figs18"] = d
    return R


GROUP_VARIANTS = {"D-a (adopted)": MAIN, "D-a, one row per cell": MAIN}
GROUP_VARIANTS.update({f"D-b, Y-{Y}": classify(UNITS[(Y, "b")], "a") for Y in YS})
GV = {name: group_tests(G, distinct=(name == "D-a, one row per cell")) for name, G in GROUP_VARIANTS.items()}
for claim in ("figs2", "fig3a", "fig3b_figs13", "fig4a_figs14", "figs16ab", "figs17d", "figs18"):
    print(f"--- {claim}")
    print(pd.DataFrame({k: v[claim] for k, v in GV.items()}).to_string(float_format=lambda v: f"{v:.3g}"))


**Result (group readings): with the D-b groups or one row per cell the verdicts stay the same, except figs13 under Y-b.** With the D-b groups (109, 113 and 108 per group under Y-a, Y-b, Y-c): figs2, peak positions do not differ (KS p 0.45 to 0.93). fig3a, plastic units are higher in in/out ratio (p 0.011 to 0.018) and information per spike (p 0.007 to 0.009), but not in coherence (p 0.11 to 0.27) or half stability (p 0.16 to 0.59), nor in information per second under Y-a and Y-b (p 0.066, 0.17). fig3b, the plastic per-lap trend is not significant (p(r > 0) 0.18 to 0.35). figs13, the plastic out-of-field rate falls (p(<0) 0.017 to 0.033) and the rigid one does not (0.054 to 0.17), but under Y-b the plastic overall rate also falls (two-sided p 0.047), which the claim excludes, so figs13 fails with those groups; its outcome rests on the D-a groups. fig4a, session-wide offline p 0.14 to 0.16 and POST p 0.17 to 0.55. figs14, POST gain p 0.095 to 0.13 and change in gain p 0.25 to 0.68 (two-sided); POST bursting p 0.007 to 0.014, change in bursting p 0.41 to 0.46. figs16ab, plastic ripple-triggered rate rises (p 0.0005 to 0.038), rigid does not (0.25 to 0.58), and the coactivity change is larger in plastic cells under reading (d) (two-sided p 0.0003 to 0.003), not (c) (0.21 to 0.64). figs17d, PRE-RUN and PRE-POST differ (p 0.002 to 0.018 and 1.5e-06 to 0.003), RUN-POST does not (p 0.70 to 0.76). figs18, rigid PRE-POST and plastic RUN-POST are significant, but plastic PRE-POST is also significant (p 0.037 to 0.049) and, under Y-a, rigid RUN-POST (p 0.008). Counting the D-a units once per cell (101 rigid, 97 plastic) leaves fig4a session-wide offline at p = 0.090 and POST at p = 0.26, POST gain at p = 0.039 with change in gain p = 0.091, and the bursting change at p = 0.90.

## low-high-rate-split

> Splitting place cells into equal halves by offline firing rate gives low-rate cells with median 0.39 Hz and high-rate cells with median 1.12 Hz.

Source: Results, paragraph 5.

Criterion (claims file): The Figure S16 legend points to the Methods for the split, which the Methods do not give. From the text, place cells are split into equal groups by "off-line" sleep firing rate, and the Methods state that cells removed from an analysis are chosen within each session. Readings (a) median split within each session and running direction; (b) median split within each session; (c) a pooled median split; and for the rate, (i) offline PRE and POST together; (ii) offline PRE only. Reproduced when both medians are within tolerance under one reading. The reading(s) that match fix the split for the claims that depend on this one.

In [ ]:
OFF["gain_sess"] = np.nan
for s, r in zip(SESSIONS, RES):
    pre, post = hc11.yoke(s, "a")
    both = np.vstack([pre, post])
    rip = ripples_in(s, r, both)
    Tr = float(np.sum(rip[:, 1] - rip[:, 0]))
    for c in s["pyr"]:
        sp = s["spikes"][c]
        n_in = int(np.sum(np.searchsorted(sp, rip[:, 1]) - np.searchsorted(sp, rip[:, 0])))
        k = (OFF.session == s["name"]) & (OFF.cell == c)
        OFF.loc[k, "gain_sess"] = (n_in / Tr) / OFF.loc[k, "rate_sess_off"]
UNIT_OFF = SP[["session", "animal", "cell", "dir", "shank"]].merge(OFF, on=["session", "cell"])


def halves(df, col, by):
    """Equal halves by col within groups 'by' (the middle unit dropped when odd) [S]."""
    lab = pd.Series("", index=df.index)
    for _, g in (df.groupby(by) if by else [(None, df)]):
        g = g.sort_values(col)
        n = len(g) // 2
        lab[g.index[:n]] = "low"
        lab[g.index[len(g) - n:]] = "high"
    return lab


SPLITS = {}
rows = []
cells_df = UNIT_OFF.drop_duplicates(["session", "cell"])
for split, (df, by) in {"a session x direction": (UNIT_OFF, ["session", "dir"]),
                        "b session": (cells_df, ["session"]), "c pooled": (cells_df, None)}.items():
    for epoch, col in (("i PRE+POST", "rate_sess_off"), ("ii PRE", "rate_pre")):
        lab = halves(df, col, by)
        key = (split[0], epoch.split()[0])
        tab = df.assign(rgroup=lab)[["session", "cell", "dir", "rgroup"]] if split[0] == "a" else \
            df.assign(rgroup=lab)[["session", "cell", "rgroup"]]
        SPLITS[key] = tab
        lo, hi = df[col][lab == "low"], df[col][lab == "high"]
        rows.append(dict(split=split, rate=epoch, n_low=len(lo), n_high=len(hi), median_low=lo.median(), median_high=hi.median(),
                         match=(abs(lo.median() - 0.39) <= 0.039) & (abs(hi.median() - 1.12) <= 0.112)))
lh = pd.DataFrame(rows)
print(lh.to_string(float_format=lambda v: f"{v:.3f}"))
print("expected medians 0.39 +- 0.039 and 1.12 +- 0.112 Hz")
MATCH = [(r.split[0], r.rate.split()[0]) for r in lh.itertuples() if r.match]
SPLIT_KEY = MATCH[0] if MATCH else ("a", "i")
print("readings that match:", MATCH, "; used below:", SPLIT_KEY, "(the first listed that matches, or (a, i) if none)")


def with_rate_group(df, key=None):
    key = key or SPLIT_KEY
    tab = SPLITS[key]
    on = ["session", "cell", "dir"] if key[0] == "a" else ["session", "cell"]
    return df.merge(tab, on=on)

**Result: reproduced.** All six readings put both medians within tolerance; the first, a median split within session and direction on offline PRE and POST rate, gives 0.395 Hz for low-rate and 1.093 Hz for high-rate units (0.39 +- 0.039 and 1.12 +- 0.112). That reading fixes the split below.

## figs16c-low-rate-coding

> Low-rate cells carry more spatial information per spike (0.91 against 0.56 bits/spike), have a higher in/out place-field rate ratio (6.38 against 3.86), fewer place fields (log-mean 1.29 against 1.59), and a larger PRE to POST increase in within-ripple gain (0.71 against 0.06) than high-rate cells.

Source: Figure S16C legend; Results, paragraph 5.

Criterion (claims file): Groups from low-high-rate-split under the reading that matched there. Measures as in fig3a-plastic-more-specific and figs14-plastic-ripple-gain. "Log-mean" is read as the geometric mean. The high-rate gain change is near zero, so its tolerance is 0.07, about 10% of the gap between the groups. Reproduced when all four rank-sum tests give p < 0.05 in the stated direction and the medians are within tolerance; partially reproduced when the tests hold and a median does not.

In [ ]:
spr = with_rate_group(SP.merge(OFF[["session", "cell", "dgain"]], on=["session", "cell"]))
spr = spr[spr.rgroup != ""]
out = []
for col, alt in (("info_spike", "greater"), ("inout", "greater"), ("nfields", "less"), ("dgain", "greater")):
    x, y = spr[spr.rgroup == "low"][col].dropna(), spr[spr.rgroup == "high"][col].dropna()
    out.append(dict(measure=col, n_low=len(x), n_high=len(y), median_low=x.median(), median_high=y.median(),
                    geomean_low=np.exp(np.log(x[x > 0]).mean()), geomean_high=np.exp(np.log(y[y > 0]).mean()),
                    p_two_sided=float(stats.ranksums(x, y).pvalue),
                    p_stated_direction=float(stats.ranksums(x, y, alternative=alt).pvalue)))
print(pd.DataFrame(out).to_string(float_format=lambda v: f"{v:.4g}"))
print("expected low/high: info 0.91/0.56 bits/spike, in/out 6.38/3.86, log-mean fields 1.29/1.59, dgain 0.71/0.06")
for key in SPLITS:
    if key == SPLIT_KEY:
        continue
    t = with_rate_group(SP.merge(OFF[["session", "cell", "dgain"]], on=["session", "cell"]), key)
    t = t[t.rgroup != ""]
    print(f"split {key}: " + ", ".join(
        f"{c} low {t[t.rgroup == 'low'][c].median():.3g} / high {t[t.rgroup == 'high'][c].median():.3g} "
        f"(p {stats.ranksums(t[t.rgroup == 'low'][c].dropna(), t[t.rgroup == 'high'][c].dropna()).pvalue:.2g})"
        for c in ("info_spike", "inout", "nfields", "dgain")))

**Result: partially reproduced, resting on an assumption (split (a, i)).** All four rank-sum tests hold in the stated direction: information per spike (0.842 against 0.469 bits/spike, p = 1.1e-14), in/out ratio (6.71 against 4.05, p = 5.7e-14), number of fields (geometric mean 1.216 against 1.404, p = 1.8e-04) and change in gain (0.636 against 0.310, p = 0.0049). Within tolerance are the low-rate information, both in/out ratios and the low-rate field count; the high-rate information (0.56 +- 0.056), the high-rate field count (1.59 +- 0.159) and both gain changes (0.71 +- 0.071, 0.06 +- 0.07) are not. All six splits match in low-high-rate-split, and the claims file says the matching readings fix the split. Under the other splits (printed above) information per spike and in/out ratio go the same way, but under the three PRE-rate splits (ii) the change in gain reverses (low below high; two-sided p 0.046 under (a, ii), 0.082 under (b, ii), 0.037 under (c, ii)), so the tests do not all hold under every matching split. By the rule that the outcome is the one all readings give, this would be not reproduced; that change is proposed for the third verification round, and the verified outcome is kept here.

## fig4c-coactivation-by-rate

> In PRE sleep coactivity is dominated by fast-firing cells, whose correlations stay unchanged into POST, while slow-firing cells show the largest PRE to POST increase in coactivation and the greatest replay of maze coactivity.

Source: Figure 4B and 4C; Results, paragraph 4; Figure S16C.

Criterion (claims file): Per the Figure 4B and 4C legends, place cells sorted by session firing rate and grouped in overlapping windows of 20% of cells stepped by 1%; in each window, the mean coactivity of its pairs (different shanks) in PRE and POST, the POST - PRE change, and the partial correlation of RUN and POST coactivity given PRE. Whether ranks are taken within session or pooled is open (readings (a) percentile of rate within session; (b) pooled); "session firing rate" as in fig4a-rigid-fire-faster. The legends name no test for these panels; the Spearman and signed-rank tests below are standard choices, not the paper's. Reproduced when PRE coactivity rises with the window's rate rank, the POST - PRE change and the partial correlation fall with it (Spearman over windows, both p < 0.05), the change in the top window is not significant (signed-rank over pairs, p > 0.05), and, per Figure S16C, the low-rate half's mean change exceeds the high-rate half's (rank-sum, p < 0.05). Partially reproduced when the slow-firing increase holds and the PRE dominance or the top-window stability does not.

In [ ]:
def partial_r(x, y, z):
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    return stats.pearsonr(rx, ry)[0]


cellrate = OFF.set_index(["session", "cell"]).rate_sess_off
for reading in ("a within session", "b pooled"):
    pr = PAIRS.copy()
    allcells = pd.DataFrame(sorted({(s, c) for s, c in zip(pr.session, pr.a)} | {(s, c) for s, c in zip(pr.session, pr.b)}),
                            columns=["session", "cell"])
    allcells["rate"] = [cellrate[(s, c)] for s, c in zip(allcells.session, allcells.cell)]
    if reading[0] == "a":
        allcells["pct"] = allcells.groupby("session").rate.rank(pct=True)
    else:
        allcells["pct"] = allcells.rate.rank(pct=True)
    pct = allcells.set_index(["session", "cell"]).pct
    pr["pa"] = [pct[(s, c)] for s, c in zip(pr.session, pr.a)]
    pr["pb"] = [pct[(s, c)] for s, c in zip(pr.session, pr.b)]
    wins = []
    for lo in np.arange(0, 0.81, 0.01):
        sub = pr[(pr.pa > lo) & (pr.pa <= lo + 0.2) & (pr.pb > lo) & (pr.pb <= lo + 0.2)].dropna(subset=["co_PRE", "co_POST", "co_RUN"])
        if len(sub) < 10:
            continue
        wins.append(dict(lo=lo, n=len(sub), pre=sub.co_PRE.mean(), change=sub.dco.mean(),
                         partial=partial_r(sub.co_RUN.values, sub.co_POST.values, sub.co_PRE.values)))
    w = pd.DataFrame(wins)
    top = pr[(pr.pa > 0.8) & (pr.pb > 0.8)].dco.dropna()
    low_half = pr[(pr.pa <= 0.5) & (pr.pb <= 0.5)].dco.dropna()
    high_half = pr[(pr.pa > 0.5) & (pr.pb > 0.5)].dco.dropna()
    print(f"reading {reading}: {len(w)} windows (pairs per window {w.n.min()} to {w.n.max()})")
    for col in ("pre", "change", "partial"):
        rho, p = stats.spearmanr(w.lo, w[col])
        print(f"   {col}: Spearman rho with rate rank = {rho:+.3f}, p = {p:.3g}; lowest window {w[col].iloc[0]:.4f}, "
              f"highest {w[col].iloc[-1]:.4f}")
    print(f"   top window (top 20% of cells) change: mean {top.mean():+.4f}, n = {len(top)}, signed-rank p = "
          f"{hc11.wilcoxon_1s(top, 'two-sided'):.3g}")
    print(f"   low half change {low_half.mean():+.4f} (n = {len(low_half)}) vs high half {high_half.mean():+.4f} "
          f"(n = {len(high_half)}), rank-sum p(low > high) = {stats.ranksums(low_half, high_half, alternative='greater').pvalue:.3g}")

**Result: reproduced, resting on an assumption.** With rate ranks taken within session, PRE coactivity rises with rate rank (Spearman rho = +0.656, p = 3.0e-11), the POST - PRE change falls (rho = -0.327, p = 0.0029), the partial correlation of RUN and POST falls (rho = -0.696, p = 5.6e-13), the top window does not change (p = 0.56), and the low-rate half's change exceeds the high-rate half's (p = 0.019). With pooled ranks the first three hold but the halves do not differ (p = 0.64). The criterion names both readings, and the outcome should be the one both give: under pooled ranks the Figure S16C condition fails, so the claim holds only in part. A change to partially reproduced is proposed for the third verification round; the verified outcome is kept here.

## fig4d-slow-cells-replay

> Slow-firing, but not fast-firing, place cells increased their contribution to sequences from PRE to POST sleep.

Source: Figure 4D; Results, paragraph 5; Figures S20D, S21B and S22E.

Criterion (claims file): The Bayesian replay analysis rerun with templates of only the low-rate or only the high-rate place cells (low-high-rate-split), the event criteria applied to the kept cells. The test of the change is open; readings (a) a rank-sum test of POST against PRE rZ within each group (as in Figure 1D); (b) a bootstrapped 95% interval of mean POST - PRE rZ (as in Figure 2D). Reproduced when the low-rate group's POST - PRE rZ is above zero (p < 0.05) and the high-rate group's is not, under each reading; partially reproduced when the low-rate increase holds and the high-rate group also increases but less. The paired-latency (Figure S20D) and time-bin-shuffle (Figure S22E) versions are shown as variants.

In [ ]:
# Replay rerun on templates of subsets of place cells, the event criteria applied to the
# kept cells [M, Bayesian Replay Analysis]; 1,000 circular shuffles each.
rng_sub = np.random.default_rng(404)
gain_lab = halves(UNIT_OFF, "gain_sess" if SPLIT_KEY[1] == "i" else "gain_pre",
                  ["session", "dir"] if SPLIT_KEY[0] == "a" else ["session"]) if SPLIT_KEY[0] == "a" else None
if SPLIT_KEY[0] == "a":
    GAINTAB = UNIT_OFF.assign(ggroup=gain_lab)[["session", "cell", "dir", "ggroup"]]
else:
    cd = UNIT_OFF.drop_duplicates(["session", "cell"])
    GAINTAB = cd.assign(ggroup=halves(cd, "gain_sess" if SPLIT_KEY[1] == "i" else "gain_pre",
                                      ["session"] if SPLIT_KEY[0] == "b" else None))[["session", "cell", "ggroup"]]
RATETAB = SPLITS[SPLIT_KEY]


def subset_cells(s, d, cells, kind):
    on_dir = SPLIT_KEY[0] == "a"
    if kind in ("low", "high"):
        t = RATETAB[(RATETAB.session == s["name"]) & (RATETAB.rgroup == kind) & ((RATETAB.dir == d) if on_dir else True)]
        keep = set(t.cell)
        return [c for c in cells if c in keep]
    if kind in ("lowgain", "highgain"):
        t = GAINTAB[(GAINTAB.session == s["name"]) & (GAINTAB.ggroup == kind[:-4]) & ((GAINTAB.dir == d) if on_dir else True)]
        keep = set(t.cell)
        return [c for c in cells if c in keep]
    grp = MAIN[(MAIN.session == s["name"]) & (MAIN.dir == d)]
    drop = set(grp[grp.group == kind.replace("no_", "")].cell)
    return [c for c in cells if c not in drop]


SUB = {}
t_sub = time.time()
for kind in ("low", "high", "lowgain", "highgain", "no_rigid", "no_plastic"):
    res_k = []
    for s, r in zip(SESSIONS, RES):
        rr = dict(name=s["name"], rep={})
        for d, rep in r["rep"].items():
            cells = subset_cells(s, d, rep["cells"], kind)
            pm = r["pm"][d]
            if len(cells) < 5:
                rr["rep"][d] = dict(events=rep["events"].iloc[:0].assign(k=0), cells=cells)
                continue
            maps = np.array([pm["maps"][c] for c in cells])
            raw = np.array([pm["raw"][c] for c in cells])
            out = hc11.replay_set(s, r["ripples"], cells, maps, raw, rng_sub, n_shuf=hc11.N_SHUF)
            out["cells"] = cells
            rr["rep"][d] = out
        res_k.append(rr)
    SUB[kind] = res_k
    print(f"{kind}: {time.time() - t_sub:.0f} s", flush=True)


def post_minus_pre(results, Y="a", D="a", seed=0):
    ev = replay_events(Y, D, results=results)
    a, b = ev.rz[ev.win == "PRE"].dropna().values, ev.rz[ev.win == "POST"].dropna().values
    rng = np.random.default_rng(seed)
    boot = [b[rng.integers(0, len(b), len(b))].mean() - a[rng.integers(0, len(a), len(a))].mean() for _ in range(2000)]
    return dict(n_PRE=len(a), n_POST=len(b), mean_PRE=a.mean(), mean_POST=b.mean(), diff=b.mean() - a.mean(),
                ci_lo=np.percentile(boot, 2.5), ci_hi=np.percentile(boot, 97.5),
                p_ranksum=float(stats.ranksums(b, a, alternative="greater").pvalue))


SUBTAB = pd.DataFrame([dict(template=k, Y=Y, D=D, **post_minus_pre(SUB[k] if k != "all" else RES, Y, D, seed=7))
                       for k in ("all", "low", "high", "lowgain", "highgain", "no_rigid", "no_plastic")
                       for Y in YS for D in DS])
print(SUBTAB[SUBTAB.template.isin(["all", "low", "high"])].to_string(float_format=lambda v: f"{v:.4g}"))
print("cells per template (Y-a, D-a):", {k: sum(len(v["cells"]) for rr in SUB[k] for v in rr["rep"].values()) for k in SUB})

In [ ]:
# Variant: fig4d and fig4e under the five other rate splits (all six match in
# low-high-rate-split), with 100 circular shuffles per template in place of 1,000 to keep
# the run time (fig2d shows 100 and 1,000 templates agreeing).
def split_tabs(key):
    gcol = "gain_sess" if key[1] == "i" else "gain_pre"
    if key[0] == "a":
        gt = UNIT_OFF.assign(ggroup=halves(UNIT_OFF, gcol, ["session", "dir"]))[["session", "cell", "dir", "ggroup"]]
    else:
        cd = UNIT_OFF.drop_duplicates(["session", "cell"])
        gt = cd.assign(ggroup=halves(cd, gcol, ["session"] if key[0] == "b" else None))[["session", "cell", "ggroup"]]
    return SPLITS[key], gt


rng_sv = np.random.default_rng(405)
t_sv = time.time()
rows_sv = []
for key in SPLITS:
    if key == SPLIT_KEY:
        continue
    rt, gt = split_tabs(key)
    for kind in ("low", "high", "lowgain", "highgain"):
        tab, col, lab = (rt, "rgroup", kind) if kind in ("low", "high") else (gt, "ggroup", kind[:-4])
        res_k = []
        for s, r in zip(SESSIONS, RES):
            rr = dict(name=s["name"], rep={})
            for d, rep in r["rep"].items():
                t = tab[(tab.session == s["name"]) & (tab[col] == lab)]
                if key[0] == "a":
                    t = t[t.dir == d]
                keep = set(t.cell)
                cells = [c for c in rep["cells"] if c in keep]
                if len(cells) < 5:
                    rr["rep"][d] = dict(events=rep["events"].iloc[:0].assign(k=0), cells=cells)
                    continue
                pm = r["pm"][d]
                out = hc11.replay_set(s, r["ripples"], cells, np.array([pm["maps"][c] for c in cells]),
                                      np.array([pm["raw"][c] for c in cells]), rng_sv, n_shuf=100)
                out["cells"] = cells
                rr["rep"][d] = out
            res_k.append(rr)
        for Y in YS:
            for D in DS:
                rows_sv.append(dict(split=f"{key[0]}, {key[1]}", template=kind, Y=Y, D=D,
                                    **post_minus_pre(res_k, Y, D, seed=7)))
SV = pd.DataFrame(rows_sv)
print(SV[(SV.Y == "a") & (SV.D == "a")][["split", "template", "n_PRE", "n_POST", "diff", "ci_lo", "ci_hi", "p_ranksum"]]
      .to_string(float_format=lambda v: f"{v:.4g}"))
summ = SV.assign(sig=(SV.p_ranksum < 0.05), ci_pos=SV.ci_lo > 0).groupby(["split", "template"]).agg(
    diff_min=("diff", "min"), diff_max=("diff", "max"), readings=("sig", "size"), p_below_05=("sig", "sum"),
    ci_above_0=("ci_pos", "sum")).reset_index()
print("over the six Y x D readings (rank-sum p < 0.05 and interval above 0, counts of readings):")
print(summ.to_string(float_format=lambda v: f"{v:.4g}"))
for a, b, nm in (("low", "high", "fig4d"), ("highgain", "lowgain", "fig4e")):
    for sp_, g in SV.groupby("split"):
        x, y = g[g.template == a].reset_index(), g[g.template == b].reset_index()
        print(f"{nm} split ({sp_}): {a} increase larger than {b} in {(x['diff'] > y['diff']).sum()} of {len(x)} readings")
print(f"time {time.time() - t_sv:.0f} s")


**Result (rate splits):** fig4d, under every split the low-rate increase holds in all six Y x D readings (POST - PRE 0.040 to 0.113, rank-sum p < 0.05 and interval above zero in 6 of 6). The high-rate increase is also significant in 6 of 6 readings under (a, ii), (b, i) and (c, i), in 6 (interval 5) under (b, ii) and in 4 (interval 3) under (c, ii); at Y-a, D-a it is smallest under the PRE-rate splits (0.019, 0.017, 0.007). The low-rate increase is the larger in 6 of 6 readings under (a, ii) and (c, ii), 4 under (b, i) and (b, ii), and 3 under (c, i). No split gives the reproduced pattern. fig4e, the high-gain increase is significant in all six readings under every split. The low-gain increase is significant in none under (b, ii), in 1 (interval 2) under (a, ii), and in all six under (b, i), (c, i) and (c, ii) (interval 6, 6 and 4). Under split (b, ii) fig4e meets its reproduced criterion. These variants use 100 shuffles per template.

**Result: partially reproduced, resting on an assumption (split (a, i)).** With low-rate templates POST - PRE rZ is above zero under every reading (Y-a, D-a: 0.050, interval 0.027 to 0.073, rank-sum p = 4.2e-04). With high-rate templates it is also above zero (0.027, interval 0.008 to 0.044, p = 9.5e-04), but smaller than the low-rate increase under all six readings (0.027 to 0.065 against 0.050 to 0.078). Under the other splits (Result (rate splits) above), the high-rate increase is significant in all six readings under four splits, and it exceeds the low-rate increase in 2 of 6 readings under (b, i) and (b, ii) and 3 of 6 under (c, i), so there even the partial pattern (high-rate increase smaller) fails. By the rule that the outcome is the one all readings give, this would be not reproduced; that change is proposed for the third verification round, and the verified outcome is kept here.

## fig4e-high-gain-cells-replay

> The PRE to POST increase in sequence content was limited to cells with high ripple-specific recruitment (within-ripple rate gain).

Source: Figure 4E; Results, paragraph 5; Figures S20E, S21C and S22E.

Criterion (claims file): As fig4d-slow-cells-replay, with the place cells split into equal halves by within-ripple firing-rate gain, under the split and epoch readings that matched in low-high-rate-split. Reproduced when the high-gain group's POST - PRE rZ is above zero (p < 0.05) and the low-gain group's is not, under each reading of the test.

In [ ]:
print(SUBTAB[SUBTAB.template.isin(["all", "lowgain", "highgain"])].to_string(float_format=lambda v: f"{v:.4g}"))

**Result: partially reproduced, resting on an assumption.** With high-gain templates POST - PRE rZ is above zero under every reading (Y-a, D-a: 0.048, interval 0.026 to 0.069, p = 3.6e-06). With low-gain templates the increase is smaller (0.017) and its bootstrap interval includes zero (-0.002 to 0.036), but the rank-sum test finds it (p = 0.015); under D-b neither test finds it (rank-sum p 0.32 to 0.62).

## fig2e-plastic-removal

> The PRE to POST increase in sequence content is attributable to a small number of plastic cells, judged by removing rigid or plastic cells from the replay analysis.

Source: Figure 2E legend.

Criterion (claims file): The Figure 2E panel is not available here and its legend gives no method beyond removal. Removal as in figs12-high-dpcc-exclusion (cells excluded from template and event selection, the 100 ms and 10% or 5 cell criteria applied after exclusion, within each session, results pooled), excluding all rigid or all plastic cells of rigid-plastic-counts; mean POST rZ - mean PRE rZ with a bootstrapped 95% interval, the annotation of Figure 2D and Figure S12. Reproduced when, after removing the plastic cells, the interval of POST - PRE rZ reaches zero or below, and after removing the rigid cells it stays above zero, under readings Y and D; partially reproduced when both stay above zero but removing plastic cells lowers POST - PRE rZ more than removing rigid cells.

In [ ]:
print(SUBTAB[SUBTAB.template.isin(["all", "no_rigid", "no_plastic"])].to_string(float_format=lambda v: f"{v:.4g}"))

In [ ]:
# Variant: removal with the D-b groups (Y-a, Y-b, Y-c; 109, 113, 108 per group), with 200
# circular shuffles per template in place of 1,000 to keep the run time; each removal is
# pooled under every Y and D reading.
rng_e = np.random.default_rng(406)
t_e = time.time()
rows_e = []
for gname, G in GROUP_VARIANTS.items():
    if not gname.startswith("D-b"):
        continue
    for kind in ("no_rigid", "no_plastic"):
        res_k = []
        for s, r in zip(SESSIONS, RES):
            rr = dict(name=s["name"], rep={})
            for d, rep in r["rep"].items():
                grp = G[(G.session == s["name"]) & (G.dir == d)]
                drop = set(grp[grp.group == kind[3:]].cell)
                cells = [c for c in rep["cells"] if c not in drop]
                pm = r["pm"][d]
                out = hc11.replay_set(s, r["ripples"], cells, np.array([pm["maps"][c] for c in cells]),
                                      np.array([pm["raw"][c] for c in cells]), rng_e, n_shuf=200)
                out["cells"] = cells
                rr["rep"][d] = out
            res_k.append(rr)
        for Y in YS:
            for D in DS:
                rows_e.append(dict(groups=gname, template=kind, Y=Y, D=D, **post_minus_pre(res_k, Y, D, seed=7)))
F2E = pd.DataFrame(rows_e)
print(F2E[["groups", "template", "Y", "D", "n_PRE", "n_POST", "diff", "ci_lo", "ci_hi", "p_ranksum"]]
      .to_string(float_format=lambda v: f"{v:.4g}"))
summ_e = F2E.assign(ci_reaches_0=F2E.ci_lo <= 0).groupby(["groups", "template"]).agg(
    diff_min=("diff", "min"), diff_max=("diff", "max"), readings=("diff", "size"),
    interval_reaches_0=("ci_reaches_0", "sum")).reset_index()
print("over the six Y x D readings:")
print(summ_e.to_string(float_format=lambda v: f"{v:.4g}"))
print(f"time {time.time() - t_e:.0f} s")


**Result: reproduced, resting on an assumption.** Removing the plastic units turns POST - PRE rZ negative (Y-a, D-a: -0.019, interval -0.036 to -0.002; the interval reaches zero or below under every reading), and removing the rigid units leaves it above zero (0.124, interval 0.108 to 0.141; above zero under every reading). These use the D-a groups. With the D-b groups (cell above, 200 shuffles per template), removing the plastic units leaves the interval above zero in all six readings with the Y-a groups (0.031 to 0.044), in four of six with the Y-b groups and in two of six with the Y-c groups; removing the rigid units leaves it above zero in every reading (0.089 to 0.170). Removing plastic units lowers the change more than removing rigid units in every reading, the partial pattern. The criterion asks for the result under readings Y and D, so by the rule that the outcome is the one all readings give, this would be partially reproduced; that change is proposed for the third verification round, and the verified outcome is kept here.

## figs16de-more-ripples-post

> Ripple events occur more often in POST than in PRE (median inter-event interval 2.47 s against 3 s), as do purely LFP-detected sharp wave-ripples (1.06 s against 1.21 s).

Source: Figure S16D and S16E (medians read from the panel titles).

Criterion (claims file): Intervals between consecutive events within offline states, intervals under 60 s only (legend); rank-sum test of PRE against POST intervals ("p ~ 0" printed, recorded as 0). Ripple events as in the opening comment; LFP-SWR events under readings L-a and L-b. Reproduced when POST intervals are shorter for both event types at p < 0.05 and the ripple-event medians are within tolerance; the LFP-SWR medians depend on the open detector and are reported, matching under one L reading counts toward reproduced.

In [ ]:
for Y in YS:
    iv_rows = []
    for s, r in zip(SESSIONS, RES):
        pre, post = hc11.yoke(s, Y)
        rp = r["ripples"]
        for w, iv in (("PRE", pre), ("POST", post)):
            st = rp.start.values
            st = st[hc11.in_intervals(st, iv)]
            # intervals between consecutive events inside the same offline interval
            k = np.searchsorted(hc11.merge(iv)[:, 0], st, side="right")
            dd = np.diff(st)
            same = np.diff(k) == 0
            v = dd[same & (dd < 60)]
            iv_rows.append(pd.DataFrame(dict(win=w, ieI=v, session=s["name"])))
    ivs = pd.concat(iv_rows)
    a, b = ivs.ieI[ivs.win == "PRE"], ivs.ieI[ivs.win == "POST"]
    print(f"Y-{Y}: ripple-event intervals PRE median {a.median():.3f} s (n = {len(a)}), POST {b.median():.3f} s (n = {len(b)}), "
          f"rank-sum p(POST shorter) = {stats.ranksums(b, a, alternative='less').pvalue:.3g}")
    if Y == "a":
        print(ivs.groupby(["session", "win"]).ieI.median().unstack().round(2).to_string())
print("expected ripple-event medians PRE 3 +- 0.3 s, POST 2.47 +- 0.247 s; the LFP-SWR intervals are not tested (no LFP read)")

**Result: not reproduced.** Ripple events come more often in POST than in PRE (median interval 1.242 against 1.348 s, rank-sum p = 2.4e-17 under Y-a; p < 1e-16 under Y-b and Y-c), but both medians are less than half the paper's (3 +- 0.3 and 2.47 +- 0.247 s), which the criterion requires. These events lack the LFP condition, so they are not the claim's events, and the LFP-SWR intervals were not tested (the LFP was not read).

## fig2d-high-dpcc-shuffle-abolishes-replay

> Shuffling the place fields of the 10 to 20% of cells with the highest dPCC eliminates the POST over PRE increase in sequence content, while it survives shuffling of the cells with the lowest dPCC.

Source: Figure 2D; Results, paragraph 2.

Criterion (claims file): Per the Methods (Per Cell Contribution, last paragraph) and the Figure 2D legend, within each session an increasing percentage of place cells (ranked by dPCC, from the highest or from the lowest) has its maps circularly shuffled (1,000 templates), the replay analysis is rerun, and mean POST rZ - mean PRE rZ is pooled over sessions, with a bootstrapped 95% interval. Percentage steps are not given; 5% steps are a standard choice. The expected value and tolerance stand for the range 10 to 20%, widened by 10% of each bound (9 to 22%). Reproduced when, shuffling from the highest dPCC, the change first falls to zero or below between 9 and 22%, and, shuffling from the lowest dPCC, it stays above zero up to 75%, under readings Y and D.

In [ ]:
# Units ranked by dPCC (Y-a, D-a) within each session; an increasing percentage, from the
# highest or the lowest, has its maps circularly shuffled [M, PCC Analysis, last paragraph].
N_TPL = 100  # partially shuffled templates per step (the paper uses 1,000; see the check below)
STEPS = {"high": np.arange(0, 31, 5), "low": np.arange(0, 76, 5)}  # the ranges the claim is judged on
U0 = UNITS[("a", "a")]


def ranked_units(sname, order):
    u = U0[U0.session == sname].sort_values("dpcc", ascending=(order == "low"))
    return list(zip(u.cell, u.dir))


def mean_rz_partial(s, r, d, shuffled, n_tpl, rng):
    rep = r["rep"][d]
    ev = rep["events"]
    cells = rep["cells"]
    pm = r["pm"][d]
    maps = np.array([pm["maps"][c] for c in cells])
    raw = np.array([pm["raw"][c] for c in cells])
    S, evb, nb = hc11.bin_counts(s, cells, ev.start.values, ev.stop.values)
    tb = np.concatenate([np.arange(k) for k in nb]).astype(np.float32)
    sel = np.array([c in shuffled for c in cells])
    if not sel.any():
        return ev.rz.values.copy()
    P = maps.shape[1]
    shifts = rng.integers(0, P, size=(n_tpl, sel.sum()))

    def one(sh):
        F = maps.copy()
        F[sel] = hc11.rotate_smooth(raw[sel], sh)
        F = F + hc11.RATE_FLOOR
        a, b, _ = hc11.decode_moments(S, np.log(F).astype(np.float32), F.sum(0).astype(np.float32))
        return (np.abs(hc11._wcorr_from_moments(a, b, evb, nb, tb)) - ev.null_mu.values) / ev.null_sd.values

    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(hc11.N_THREADS) as ex:
        return np.nanmean(np.stack(list(ex.map(one, shifts)), 1), 1)


def pooled_change(rz_by_sd, seed, Y="a", D="a", n_boot=1000):
    pre, post = [], []
    for (sname, d), rz in rz_by_sd.items():
        s = SES[sname]
        ev = RESD[sname]["rep"][d]["events"]
        p0, p1 = hc11.yoke(s, Y)
        mid = (ev.start.values + ev.stop.values) / 2
        keep = (ev.assigned.values == d) if D == "b" else np.ones(len(ev), bool)
        pre.append(rz[hc11.in_intervals(mid, p0) & keep])
        post.append(rz[hc11.in_intervals(mid, p1) & keep])
    a, b = np.concatenate(pre), np.concatenate(post)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    rng = np.random.default_rng(seed)
    boot = [b[rng.integers(0, len(b), len(b))].mean() - a[rng.integers(0, len(a), len(a))].mean() for _ in range(n_boot)]
    return b.mean() - a.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5)


t2d = time.time()
rng2d = np.random.default_rng(2)
rows2d = []
STORE2D = {}
for order in ("high", "low"):
    for pctv in STEPS[order]:
        rz_by_sd = {}
        for s, r in zip(SESSIONS, RES):
            ranked = ranked_units(s["name"], order)
            k = int(round(pctv / 100 * len(ranked)))
            chosen = set(ranked[:k])
            for d in r["rep"]:
                rz_by_sd[(s["name"], d)] = mean_rz_partial(s, r, d, {c for c, dd in chosen if dd == d}, N_TPL, rng2d)
        diff, lo, hi = pooled_change(rz_by_sd, seed=int(pctv))
        rows2d.append(dict(order=order, pct_shuffled=pctv, post_minus_pre=diff, ci_lo=lo, ci_hi=hi))
        STORE2D[(order, pctv)] = rz_by_sd
F2D = pd.DataFrame(rows2d)
print(F2D.pivot(index="pct_shuffled", columns="order", values=["post_minus_pre", "ci_lo", "ci_hi"]).round(4).to_string())
hi_ = F2D[F2D.order == "high"]
first_zero = hi_[hi_.post_minus_pre <= 0].pct_shuffled.min()
first_ci0 = hi_[hi_.ci_lo <= 0].pct_shuffled.min()
lo_ = F2D[F2D.order == "low"]
print(f"from the highest dPCC: POST - PRE first <= 0 at {first_zero}% shuffled; its 95% interval first reaches 0 at {first_ci0}%")
print(f"from the lowest dPCC: POST - PRE above 0 at every step up to 75%: {bool((lo_[lo_.pct_shuffled <= 75].post_minus_pre > 0).all())}; "
      f"lower bound above 0 up to 75%: {bool((lo_[lo_.pct_shuffled <= 75].ci_lo > 0).all())}")
# check of the number of templates: 1,000 templates at 15% from the highest dPCC
rz_by_sd = {}
for s, r in zip(SESSIONS, RES):
    ranked = ranked_units(s["name"], "high")
    chosen = set(ranked[:int(round(0.15 * len(ranked)))])
    for d in r["rep"]:
        rz_by_sd[(s["name"], d)] = mean_rz_partial(s, r, d, {c for c, dd in chosen if dd == d}, 1000, rng2d)
print("check, 15%% from the highest with 1,000 templates: POST - PRE = %.4f (95%% interval %.4f to %.4f)" % pooled_change(rz_by_sd, 15))
print(f"time {time.time() - t2d:.0f} s")

In [ ]:
# Variant: the same partially shuffled templates (ranked by dPCC under Y-a, D-a), with the
# change pooled under the other Y and D readings; D-b keeps each event only on the template
# of the direction it is assigned to (equation 6).
rows = []
for Y in YS:
    for D in DS:
        for (order, pctv), rz_by_sd in STORE2D.items():
            diff, lo, hi = pooled_change(rz_by_sd, seed=int(pctv), Y=Y, D=D, n_boot=500)
            rows.append(dict(Y=Y, D=D, order=order, pct_shuffled=pctv, post_minus_pre=diff, ci_lo=lo, ci_hi=hi))
F2DV = pd.DataFrame(rows)
summ = []
for (Y, D), g in F2DV.groupby(["Y", "D"]):
    h, l = g[g.order == "high"], g[g.order == "low"]
    summ.append(dict(Y=Y, D=D, high_first_le0=h[h.post_minus_pre <= 0].pct_shuffled.min(),
                     high_at_first=h[h.post_minus_pre <= 0].post_minus_pre.iloc[0] if (h.post_minus_pre <= 0).any() else np.nan,
                     low_min_change=l.post_minus_pre.min(), low_all_above0=bool((l.post_minus_pre > 0).all()),
                     low_lower_bound_min=l.ci_lo.min()))
print(pd.DataFrame(summ).to_string(float_format=lambda v: f"{v:.4f}"))
print(F2DV[F2DV.order == "high"].pivot(index="pct_shuffled", columns=["Y", "D"], values="post_minus_pre").round(4).to_string())


**Result (Y and D readings):** with the same ranking, shuffling from the highest dPCC first takes the change to zero or below at 10% under four readings and at 15% under Y-b, D-a and Y-c, D-b (all inside 9 to 22%); from the lowest it stays above zero up to 75% under all six readings (smallest lower bound 0.063). The cells are ranked by dPCC under Y-a, D-a in every reading.

**Result: reproduced, resting on an assumption.** Shuffling from the highest dPCC, POST - PRE rZ falls from 0.096 to 0.045 at 5% and first reaches zero or below at 10% (-0.003, interval -0.017 to 0.011), inside 9 to 22%; with 1,000 templates the 15% step gives -0.020 (100 templates: -0.020). Shuffling from the lowest dPCC, it stays above zero up to 75% (0.070 at 75%, lower bound 0.063). Cells are ranked by dPCC under Y-a, D-a; the change pooled under all six Y and D readings is in the cell above.

## figs12-high-dpcc-exclusion

> Excluding the 10 to 20% of cells with the highest dPCC turns the POST minus PRE change in sequence content negative, while excluding up to about 75% of the lowest dPCC cells leaves POST above PRE.

Source: Figure S12.

Criterion (claims file): As fig2d-high-dpcc-shuffle-abolishes-replay, with cells excluded from the template and event selection (the 100 ms and 10% or 5 cell criteria applied after exclusion, per the Methods) in place of shuffled. Reproduced when, excluding from the highest dPCC, POST - PRE rZ falls below zero between 9 and 22% excluded, and, excluding from the lowest, the lower bound of its bootstrapped 95% interval stays above zero up to 75%, under readings Y and D.

In [ ]:
# Cells excluded from the template and from the event criteria, within each session; the
# null is rebuilt from 200 circular shuffles per step (the paper uses 1,000).
N_EXCL_SHUF = 200
t12 = time.time()
rng12 = np.random.default_rng(12)
rows12 = []
STORE12 = {}
for order, steps in (("high", np.arange(0, 31, 5)), ("low", np.r_[0:71:10, 75])):
    for pctv in steps:
        rz_by_sd = {}
        for s, r in zip(SESSIONS, RES):
            ranked = ranked_units(s["name"], order)
            drop = set(ranked[:int(round(pctv / 100 * len(ranked)))])
            for d, rep in r["rep"].items():
                cells = [c for c in rep["cells"] if (c, d) not in drop]
                if len(cells) < 5:
                    continue
                pm = r["pm"][d]
                out = hc11.replay_set(s, r["ripples"], cells, np.array([pm["maps"][c] for c in cells]),
                                      np.array([pm["raw"][c] for c in cells]), rng12, n_shuf=N_EXCL_SHUF)
                rz_by_sd[(s["name"], d)] = out
        pre, post = [], []
        for (sname, d), out in rz_by_sd.items():
            ev = out["events"]
            p0, p1 = hc11.yoke(SES[sname], "a")
            mid = (ev.start.values + ev.stop.values) / 2
            pre.append(ev.rz.values[hc11.in_intervals(mid, p0)])
            post.append(ev.rz.values[hc11.in_intervals(mid, p1)])
        a, b = np.concatenate(pre), np.concatenate(post)
        a, b = a[~np.isnan(a)], b[~np.isnan(b)]
        rb = np.random.default_rng(int(pctv))
        boot = [b[rb.integers(0, len(b), len(b))].mean() - a[rb.integers(0, len(a), len(a))].mean() for _ in range(1000)]
        rows12.append(dict(order=order, pct_excluded=pctv, templates=len(rz_by_sd), n_PRE=len(a), n_POST=len(b),
                           post_minus_pre=b.mean() - a.mean(), ci_lo=np.percentile(boot, 2.5), ci_hi=np.percentile(boot, 97.5)))
        STORE12[(order, pctv)] = [(sname, d, (out["events"].start.values + out["events"].stop.values) / 2,
                                   out["events"].assigned.values, out["events"].rz.values) for (sname, d), out in rz_by_sd.items()]
F12 = pd.DataFrame(rows12)
print(F12.to_string(float_format=lambda v: f"{v:.4f}"))
h12 = F12[F12.order == "high"]
l12 = F12[F12.order == "low"]
print(f"from the highest dPCC: POST - PRE first < 0 at {h12[h12.post_minus_pre < 0].pct_excluded.min()}% excluded")
print(f"from the lowest dPCC: lower bound of the 95% interval above 0 at every step up to 75%: {bool((l12.ci_lo > 0).all())}")
print(f"time {time.time() - t12:.0f} s")

In [ ]:
# Variant: the same exclusions (ranked by dPCC under Y-a, D-a), with the change pooled under
# the other Y and D readings; D-b keeps each event only on the template of its assigned direction.
def pooled12(store, Y, D, seed, n_boot=500):
    pre, post = [], []
    for sname, d, mid, assigned, rz in store:
        p0, p1 = hc11.yoke(SES[sname], Y)
        keep = (assigned == d) if D == "b" else np.ones(len(rz), bool)
        pre.append(rz[hc11.in_intervals(mid, p0) & keep])
        post.append(rz[hc11.in_intervals(mid, p1) & keep])
    a, b = np.concatenate(pre), np.concatenate(post)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    rb = np.random.default_rng(seed)
    boot = [b[rb.integers(0, len(b), len(b))].mean() - a[rb.integers(0, len(a), len(a))].mean() for _ in range(n_boot)]
    return b.mean() - a.mean(), np.percentile(boot, 2.5), np.percentile(boot, 97.5)


rows = []
for Y in YS:
    for D in DS:
        for (order, pctv), store in STORE12.items():
            diff, lo, hi = pooled12(store, Y, D, int(pctv))
            rows.append(dict(Y=Y, D=D, order=order, pct_excluded=pctv, post_minus_pre=diff, ci_lo=lo, ci_hi=hi))
F12V = pd.DataFrame(rows)
summ = []
for (Y, D), g in F12V.groupby(["Y", "D"]):
    h, l = g[g.order == "high"], g[g.order == "low"]
    summ.append(dict(Y=Y, D=D, high_first_below0=h[h.post_minus_pre < 0].pct_excluded.min(),
                     low_lower_bound_min=l.ci_lo.min(), low_lower_bound_above0=bool((l.ci_lo > 0).all())))
print(pd.DataFrame(summ).to_string(float_format=lambda v: f"{v:.4f}"))
print(F12V[F12V.order == "high"].pivot(index="pct_excluded", columns=["Y", "D"], values="post_minus_pre").round(4).to_string())


**Result (Y and D readings):** with the same exclusions, the change first falls below zero at 15% under all six readings, and the lower bound stays above zero up to 75% (smallest 0.080). The cells are ranked by dPCC under Y-a, D-a in every reading.

**Result: reproduced, resting on an assumption.** Excluding from the highest dPCC, POST - PRE rZ is 0.020 at 10% and first falls below zero at 15% (-0.011), inside 9 to 22%. Excluding from the lowest dPCC, the lower bound of the 95% interval stays above zero at every step to 75% (0.176 at 75%). Cells are ranked by dPCC under Y-a, D-a, with 200 null shuffles per step; the change pooled under all six Y and D readings is in the cell above.

## figs11-cluster-quality-similar

> Rigid and plastic cells have similar waveforms and similar cluster quality and stability across PRE, MAZE and POST, so clustering errors do not explain their differences.

Source: Figure S11; Results, paragraph 3. Criterion: rank-sum tests of rigid against plastic cells at p < 0.05 (Figure S11 legend); quality is the median Mahalanobis distance of each spike's PCA features to its cluster mean, for spikes in PRE ripple events, MAZE running and POST ripple events; stability is the symmetric Mahalanobis distance between a cell's clusters in two conditions; first spikes of bursts (ISI under 6 ms) only. Reproduced when no comparison is significant.

The verified outcome of this claim is blocked (it was not tested before verification round 2); this section is the analysis behind a proposed change of outcome, for a third verification round. The features (`.fet`) and cluster labels (`.clu`) of every shank come from the eight `<session>.tar.gz` archives, streamed in a background thread since the first cell (`hc11.cluster_quality`). Waveform similarity, the first part of the statement, needs the `_spk.tar.gz` archives (87 GB) and is not tested; the criterion judges quality and stability only. Choices, stated in `hc11.py`: the PCA features are the first three per channel of each `.fet` row; the covariance of the Mahalanobis distance is that of all the cluster's kept spikes; a cell needs 10 spikes in a condition. "Only the first spike of bursts" is read two ways: (a) every spike not preceded by one within 6 ms (single spikes and first spikes of bursts), (b) only spikes that start a burst. PRE and POST ripple events are those of the yoked windows (Y-a). Groups are the adopted D-a groups, the D-a units counted once per cell, and the D-b groups.

In [ ]:
CQ_LIST, CQ_SECONDS = CQ_FUTURE.result()
CQ = pd.concat([q for q, _ in CQ_LIST], ignore_index=True)
CQI = pd.concat([i for _, i in CQ_LIST], ignore_index=True)
print(f"{len(CQI)} shanks streamed in {CQ_SECONDS / 60:.1f} min (background); .clu rows equal .fet rows on every shank: "
      f"{bool(CQI.clu_matches.all())}")
print("PCA features per shank (3 per channel of the shank's group in the .xml):", CQI.groupby("session").n_pca.agg(lambda v: sorted(set(v))).to_dict())
print(f"pyramidal cells: {len(CQ)}; cells whose cluster (id % 100 on shank id // 100) has exactly the sessInfo "
      f"spike count: {(CQ.n_fet == CQ.n_sessinfo).sum()}")
for rule in ("a", "b"):
    print(f"burst reading {rule}: median kept spikes per cell, all {CQ[f'n_all_{rule}'].median():.0f}, PRE events "
          f"{CQ[f'n_PRE_{rule}'].median():.0f}, MAZE running {CQ[f'n_MAZE_{rule}'].median():.0f}, POST events "
          f"{CQ[f'n_POST_{rule}'].median():.0f}; cells with >= 10 spikes in PRE/MAZE/POST: "
          + "/".join(str(int(CQ[f"q_{c}_{rule}"].notna().sum())) for c in ("PRE", "MAZE", "POST")))
MEASURES11 = ["q_PRE", "q_MAZE", "q_POST", "s_PRE-MAZE", "s_MAZE-POST", "s_PRE-POST"]
rows11 = []
for gname, G in GROUP_VARIANTS.items():
    gg = G[G.group != ""][["session", "cell", "dir", "group"]]
    if gname == "D-a, one row per cell":
        cg = cell_groups(G)
        gg = gg.assign(cg=[cg[(s_, c_)] for s_, c_ in zip(gg.session, gg.cell)])
        gg = gg[gg.cg != "mixed"].drop_duplicates(["session", "cell"])
    m = gg.merge(CQ, on=["session", "cell"])
    for rule in ("a", "b"):
        for meas in MEASURES11:
            col = f"{meas}_{rule}"
            if col not in m:
                continue
            x, y = m[m.group == "rigid"][col].dropna(), m[m.group == "plastic"][col].dropna()
            rows11.append(dict(groups=gname, rule=rule, measure=meas, n_rigid=len(x), n_plastic=len(y),
                               median_rigid=x.median(), median_plastic=y.median(), diff=y.median() - x.median(),
                               p=float(stats.ranksums(x, y).pvalue) if min(len(x), len(y)) > 1 else np.nan))
FIG11 = pd.DataFrame(rows11)
print(FIG11.to_string(float_format=lambda v: f"{v:.4g}"))
print("comparisons with rank-sum p < 0.05, per group set and burst reading:")
print(FIG11.assign(sig=FIG11.p < 0.05).groupby(["groups", "rule"]).agg(
    comparisons=("p", "size"), significant=("sig", "sum"), smallest_p=("p", "min")).to_string(float_format=lambda v: f"{v:.3g}"))
m = MAIN[MAIN.group != ""].merge(CQ, on=["session", "cell"])
print("by animal (D-a groups, reading a), median quality PRE / MAZE / POST:")
print(m.groupby(["animal", "group"])[["q_PRE_a", "q_MAZE_a", "q_POST_a"]].median().round(3).to_string())
print("range of the quality and stability values (reading a):",
      f"{np.nanmin(CQ[[f'{c}_a' for c in MEASURES11]].values):.2f} to {np.nanmax(CQ[[f'{c}_a' for c in MEASURES11]].values):.2f}")


**Result (proposed change; the verified outcome stays blocked): reproduced, resting on an assumption.** All 118 shanks were read; every `.clu` file has one label per `.fet` row, and each of the 562 pyramidal cells (id % 100 on shank id // 100) has exactly its sessInfo spike count, so the cells are the same. With the D-a groups and burst reading (a), no comparison of rigid and plastic units is significant: quality in PRE events (median 5.534 against 5.525, p = 0.83), MAZE running (5.501 against 5.440, p = 0.25) and POST events (5.573 against 5.552, p = 1.00), and stability PRE-MAZE (p = 0.98), MAZE-POST (p = 0.82) and PRE-POST (5.622 against 5.629, p = 0.13); the medians differ by 0.13 or less over all 60. None of the 60 comparisons over both burst readings and the five group sets is significant (smallest p 0.126). Under reading (b) only 283 cells have 10 or more spikes in MAZE running, so fewer units enter the MAZE comparisons (87 and 90 with the D-a groups). The number of PCA features differs between shanks (12 to 30), which shifts the distances; rigid and plastic units are split within each session, so both groups draw on the same shanks. Waveform similarity, part of the statement but not of the criterion, was not tested (the waveforms are in the `_spk.tar.gz` archives, 87 GB). The covariance and feature choices are assumptions stated in `hc11.py`.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| place-cell-count | partially reproduced (assumption) |
| fig1c-sequence-content | reproduced (assumption) |
| figs5b-abs-r-above-null | reproduced (assumption) |
| figs5e-duration-unchanged | reproduced |
| figs5f-more-participation-post | reproduced |
| figs9b-pcc-post-higher | partially reproduced (assumption) |
| fig2b-strong-contributors | partially reproduced |
| fig2-pre-contributors-persist | reproduced |
| rigid-plastic-counts | partially reproduced (assumption) |
| fig2d-high-dpcc-shuffle-abolishes-replay | reproduced (assumption) |
| figs12-high-dpcc-exclusion | reproduced (assumption) |
| figs2-rigid-plastic-along-track | reproduced (assumption) |
| fig3a-plastic-more-specific | partially reproduced (assumption) |
| fig3b-plastic-specificity-increases | not reproduced (assumption) |
| figs13-outfield-rate-falls | reproduced (assumption) |
| fig4a-rigid-fire-faster | not reproduced (assumption) |
| figs14-plastic-ripple-gain | not reproduced (assumption) |
| figs14-plastic-ripple-bursting | not reproduced (assumption) |
| figs16ab-plastic-synchrony-increase | reproduced (assumption) |
| low-high-rate-split | reproduced |
| figs16c-low-rate-coding | partially reproduced (assumption); proposed: not reproduced |
| fig4c-coactivation-by-rate | reproduced (assumption); proposed: partially reproduced |
| fig4d-slow-cells-replay | partially reproduced (assumption); proposed: not reproduced |
| fig4e-high-gain-cells-replay | partially reproduced (assumption) |
| figs16de-more-ripples-post | not reproduced |
| figs17-coactivity-preservation | partially reproduced |
| figs17d-rigid-plastic-coactivity | not reproduced (assumption) |
| figs18-temporal-bias | not reproduced (assumption) |
| figs11-cluster-quality-similar | blocked; proposed: reproduced (assumption) |
| fig2e-plastic-removal | reproduced (assumption); proposed: partially reproduced |
| figs19c-pcc-change-per-cell | not reproduced |

How the analysis differs from the paper's: ripple events lack the LFP-ripple condition (the LFP was not read), which keeps more events and shortens the intervals between them; offline states are the data set's NREM, Drowsy and Intermediate epochs, and on the maze the time not running; the rigid and plastic groups, and every claim that uses them, rest on readings Y-a, D-a and P-a; the dPCC shuffle analysis uses 100 templates per step and the exclusion analysis 200 null shuffles per step, both ranked under Y-a and D-a, with the change pooled under every Y and D reading; the rigid and plastic claims are repeated with the D-b groups, and fig4d and fig4e under every rate split (100 shuffles); fig2e is repeated with the D-b groups (200 shuffles); figs11 reads the cluster labels and spike features of the per-session archives, and waveforms are not compared; the time-bin-shuffle and paired-latency variants were not run; the downstream claims use the 5 cm kernel as an SD, though only the FWHM reading matches the place-cell count.

What was found about the data set: the eight sessions, four animals, 562 pyramidal cells and the three maze types match the paper. Spikes and position share a clock (median decoding error 6 to 12 cm aligned, 40 to 80 cm with spikes shifted by 10 s). In Gatsby_08022013 spikes continue 1,589 s past the end of the POST epoch and of `sessDuration`. Position is sampled at 39.06 Hz (the paper says 39.69). The state epochs mark only Wake on the maze, so immobility there is not given, and the 2D head speed of some sessions is too noisy to find it (Gatsby_08022013 has 0.1 min under 5 cm/s). The data set holds no list of the paper's place cells, ripple events, or rigid and plastic cells, and the LFP comes only in tar.gz archives of 5 to 10 GB per session. The cluster labels and features of the per-session archives match the sessInfo files cell for cell (562 of 562 spike counts equal).


Revisions: after verification round 1 two outcomes changed (rigid-plastic-counts to partially reproduced, figs16de to not reproduced). After round 2 no outcome changed; five changes are proposed for a third round, each with its analysis in the claim's section: fig4c to partially reproduced and figs16c and fig4d to not reproduced (readings the claims file names give different results), fig2e to partially reproduced (D-b groups), and figs11 from blocked to reproduced (now tested). This run of the notebook took 75.4 minutes on the author's runner (cell below), over the 60-minute limit; the copy reviewed in round 2 took 38.2 minutes. Of the additions in this revision, the fig2e D-b variant takes about 4 minutes and the figs11 archives stream in a background thread (44 minutes here) while the other analyses run.


In [ ]:
print(f"total run time {(time.time() - T_START) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")